# The bank of KFs, and the three banks at larger $q$, with heavy-tailed and with Gaussian noise

Leszek's four requests after the meeting of 2026-10-06, where `bank-kf-excess-error.ipynb` (commit `f2ded06`) was shown:

1. A bank of full-covariance KFs. It is not in the draft, so its equations and its algorithm, in the form of
   Algorithms 1 and 2, come first: `kf-bank-algorithm.tex`, written for Section 4.3 of `main.tex` (Overleaf `fc54c65`).
   Then the code (section 3.1).
2. The three banks (sKF, fKF and KF) at larger values of $q$, at least four: do they do better when they share more?
3. The median over the realisations instead of the mean: is it too noisy?
4. Gaussian noise ($\beta^* = 2$) beside the heavy-tailed $\beta^* = 0.2$.

**Choices made with Ramiro before writing code (2026-10-07)**

| doubt | choice |
|---|---|
| the bank of KFs | Algorithm 3 of `kf-bank-algorithm.tex`. The mixing projects onto the full family, so it keeps the whole covariance of the mixture; each filter then takes the KF step (31)-(32); the weights are eq. (56) as written, with $\sigma_{t,j}^2 = \boldsymbol{x}_t^{\mathsf T}\tilde{\boldsymbol{\Sigma}}_{t,j}\boldsymbol{x}_t$. The grid of the sKF bank (six values of $\varepsilon$, $10^{-9}$ to $10^{-4}$), $v_0 = 2$ |
| its code | the $O(KM^2)$ work in two numba kernels; $\chi$, $\chi'$ and the log density are the scipy functions of the other filters, called between them. Algorithm 3 written line by line in plain numpy is the reference of the checks |
| $q$ | $10^{-4}$ (every earlier run), $10^{-3}$, $10^{-2}$, $10^{-1}$ and $1/3$. At $1/3$ an inner value of the grid moves to each neighbour as often as it stays: no persistence is left |
| runs | 3 banks × 5 values of $q$ × {AR($-0.9$), white} × {$\beta^* = 0.2$, 2}, on notebook 09's change test, beside the exact sKF, the exact fKF and the KF at $-20$ dB: at $\beta^* = 0.2$ the values tuned in the earlier notebooks, at $\beta^* = 2$ tuned again by the same search |
| Gaussian noise | the same filters, the Laplacian likelihood with $b_\eta = \mathrm{E}\lvert\eta_t\rvert$ of the Gaussian noise: only the noise changes. The same input signals and seeds |
| median | the median over the 20 realisations at each step, with the 25-75 % band. Excess error: each realisation's $s_t^2$ smoothed over 101 steps first, then the median. The tables give every number from the mean curves and from the median curves |
| speed | no dask (Augusto's PR #16), see the end of the findings. Each run is saved under `results/kf-bank-q-sweep/` as it finishes, so an interrupted run of the notebook loses one run at most; delete that folder to run everything again |
| where | this notebook; the earlier notebooks are not touched |

**Run time** 312 min on 2026-10-07, two thirds of it the KF banks (about 10 min for each rest and change run). Run again with the runs saved under `results/kf-bank-q-sweep/`, it takes a few minutes.

## 1. Imports

In [ ]:
import pickle
import time
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt
from scipy.ndimage import uniform_filter1d
from scipy.signal import lfilter
from scipy.special import gamma as G, gammaln, log_ndtr, logsumexp
from scipy.stats import gennorm
import numba as nb
import rir_generator as rir

%config InlineBackend.figure_format = 'retina'
NOTEBOOK_START = time.time()
print("imports ready")

## 2. The scenario

Section 2 of `bank-kf-excess-error.ipynb` (commit `f2ded06`), with one option added throughout: the shape `beta` of the
generalized Gaussian noise, default `BETA` = 0.2, which gives the same draws as before. With `beta` = 2 the noise is
Gaussian with the same variance, so the same SNR; the input is drawn first, so the input signals are the same too.

In [ ]:
# === copied from bank-kf-excess-error.ipynb section 2 (commit f2ded06), which copied it from bank-skf-fkf.ipynb
#     (57bd568); one option added: the noise shape beta, default BETA (the same draws as before) ===
M = 128                     # filter length / length of the impulse response
N = 96000                   # steps per run at rest; the change run has 2N
R = 20                      # independent realisations
WARMUP = 500                # AR samples discarded so the input starts stationary
BETA = 0.2                  # shape of the generalized Gaussian measurement noise
SNR_DB = 5.0                # nominal signal-to-noise ratio
VAR_THETA_0 = 2.0           # initial prior variance on each weight (ours; Leszek's is 1/M)
FS = 8000                   # sampling rate [Hz]
TARGET_DB = -20.0           # floor the single filters are tuned to
BETA_GAUSS = 2.0            # the generalized Gaussian with beta = 2 is the Gaussian
DRIFT_DB = 0.5              # settling rule of notebook 09

# === copied from leszek-setup.ipynb section 4 (commit adb148b), NOT edited ===
ROOM, T60, C_SOUND, SRC, MIC = [5, 10, 6], 0.2, 340, [1, 2.5, 2], [1, 1.5, 1]
ho = rir.generate(c=C_SOUND, fs=FS, r=MIC, s=SRC, L=ROOM, reverberation_time=T60, nsample=M).flatten()
ho = ho/np.linalg.norm(ho)
lags = np.abs(np.subtract.outer(np.arange(M), np.arange(M)))


def noise_of(ar_a, beta=BETA):
    # Noise scale and b_eta = E|eta| at 5 dB for this input: notebook 09's lines, AR as a parameter.
    P_signal = ho @ (ar_a**lags if ar_a != 0 else np.eye(M)) @ ho
    var_eta = P_signal/10**(SNR_DB/10)
    scale_gg = np.sqrt(var_eta/np.exp(gammaln(3/beta) - gammaln(1/beta)))
    return scale_gg, scale_gg*np.exp(gammaln(2/beta) - gammaln(1/beta))


def signals(seed, ar_a, n, change_at=None, beta=BETA):
    # generate_signals / generate_signals_change of notebook 09 (commit 6e6dabc), with the AR
    # coefficient, the length and the instant of the flip as parameters. Same draws, same order.
    scale_gg, _ = noise_of(ar_a, beta)
    rng = np.random.default_rng(seed)
    u = np.sqrt(1 - ar_a**2)*rng.standard_normal(n + WARMUP)
    x = lfilter([1.0], [1.0, -ar_a], u)[WARMUP:]
    y = np.convolve(ho, x)[:n]
    if change_at is not None:
        y[change_at:] = -y[change_at:]
    eta = gennorm.rvs(beta, scale=scale_gg, size=n, random_state=rng)
    return x, y + eta
# === end of the copied block ===


def noise_variance(ar_a, beta=BETA):
    # The true noise variance, for the Gaussian likelihood: the variance of the generalized Gaussian.
    scale_gg, _ = noise_of(ar_a, beta)
    return scale_gg**2*np.exp(gammaln(3/beta) - gammaln(1/beta))


def stack(sigs):
    return np.array([s[0] for s in sigs]), np.array([s[1] for s in sigs])


for beta, ar_a in ((BETA, -0.9), (BETA, 0.0), (BETA_GAUSS, -0.9), (BETA_GAUSS, 0.0)):
    scale_gg, b_eta = noise_of(ar_a, beta)
    print(f"beta* = {beta:g}, AR {ar_a:>4}: var_eta = {noise_variance(ar_a, beta):.4e}, "
          f"noise scale = {scale_gg:.4e}, "
          f"b_eta = E|eta| = {b_eta:.4e}")
# === end of the copied block ===


def clean_output(X, change_at=None):
    # x_t' theta_t for every row of X, as signals() forms y: theta = h_o, and -h_o from change_at on
    Y = np.array([np.convolve(ho, x)[:X.shape[1]] for x in X])
    if change_at is not None:
        Y[:, change_at:] = -Y[:, change_at:]
    return Y


def signal_power(ar_a):
    # h_o' R_xx h_o: the line of noise_of
    return ho @ (ar_a**lags if ar_a != 0 else np.eye(M)) @ ho


# Copied from bank-skf-fkf.ipynb section 7 (commit 57bd568).
def scenario_signals(ar_a, beta=BETA):
    return (stack([signals(seed, ar_a, N, beta=beta) for seed in range(R)]),
            stack([signals(seed, ar_a, 2*N, change_at=N, beta=beta) for seed in range(R)]))

## 3. The filters

Section 3 of `bank-kf-excess-error.ipynb` (commit `f2ded06`), nothing edited: `log_mills`, `chi_laplacian`,
`chi_gaussian`, `sKF_batch`, `fKF_batch`, `KF_batch`, `transition_matrix`, `log_density_laplacian`, `bank_step`,
`bank_batch` and the grids. Added below it: `log_density_gaussian` from `bank-skf-fkf.ipynb` (commit `57bd568`), for
the Gaussian checks.

In [ ]:
# === copied from bank-kf-excess-error.ipynb section 3 (commit f2ded06), NOT edited; log_density_gaussian added at the end ===
# === IGNACIO: log_mills / chi_laplacian - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/07_skf_conjunto.ipynb, commit 9406077,
#         via bank-skf-fkf.ipynb (commit 57bd568)
def log_mills(z):
    """log R(z), with R(z) = Phi(-z)/phi(z) the Mills ratio, eq. (40). R grows like e^{z^2/2} for
    negative z and overflows, so it is only ever handled through its logarithm."""
    return log_ndtr(-z) + 0.5*z**2 + 0.5*np.log(2*np.pi)


def chi_laplacian(e, sigma, b_eta):
    """Correction chi_t(e_t) and its slope chi'_t(e_t) for Laplacian noise, eqs. (39), (41) and (43)."""
    u = e/sigma                                        # u_t = e_t / sigma_t
    k_t = sigma/b_eta                                  # k_t = sigma_t / b_eta
    tau = sigma**2/b_eta                               # tau_t = sigma_t^2 / b_eta, the largest correction
    log_R_minus = log_mills(k_t - u)                   # log R(k_t - u_t)
    log_R_plus = log_mills(k_t + u)                    # log R(k_t + u_t)
    Lambda = np.tanh((log_R_minus - log_R_plus)/2)     # (R- - R+)/(R- + R+), in (-1, 1)
    chi = tau*Lambda                                   # chi = tau Lambda
    chi_slope = (2*k_t*np.exp(-np.logaddexp(log_R_minus, log_R_plus))   # 2k / (R- + R+)
                 - k_t**2*(1 - Lambda**2))                               # - k^2 (1 - Lambda^2)
    return chi, chi_slope
# === end of the copied block ===


# Copied from bank-skf-fkf.ipynb (commit 57bd568), from vkf-kf.ipynb.
def chi_gaussian(e, sigma, v_eta):
    """Eq. (38), the Gaussian instance. Used only for the Kalman cross-check of section 4."""
    frac = sigma**2/(sigma**2 + v_eta)
    return frac*e, frac


# sKF_batch and fKF_batch: copied from bank-skf-fkf.ipynb (commit 57bd568), where they carry the options below;
# one option added here, `excess`, and nothing else changed.
#   flip_at: misalignment against -h from that step on (fkf.ipynb had it for the fKF only)
#   clip:    chi' -> max(chi', 0) in the variance update, eq. (36); the mean update is untouched
#   record:  also return chi'_t and v_t at every step
#   v0:      initial v (VAR_THETA_0 = 2 in every notebook of ours; 1/M in Leszek's ggbench.py)
#   wait:    first step that updates (M: wait for a full window, as every notebook of ours)
#   excess:  also return s_t = x_t'(theta_t - w_{t-1}) at every step (not with record)
def _roll_in(X_t, x_win):
    x_win = np.roll(x_win, 1, axis=1)
    x_win[:, 0] = X_t
    return x_win


def sKF_batch(X, D, h, b_eta, epsilon, chi_fn, flip_at=None, clip=False, record=False,
              v0=VAR_THETA_0, wait=None, excess=False):
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.full(B, float(v0))
    eps = np.asarray(epsilon, dtype=float)
    mis = np.empty((B, n))
    if excess:
        s = np.empty((B, n))
    if record:
        slope_hist, v_hist = np.full((B, n), np.nan), np.full((B, n), np.nan)
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_{t-1})
        if t < first:
            continue
        v_tilde = v + eps
        power = np.einsum("bm,bm->b", x_t, x_t)
        sigma = np.sqrt(v_tilde*power)
        chi, slope = chi_fn(e, sigma, b_eta)
        w = w + x_t*(chi/power)[:, None]
        v = v_tilde*(1 - (np.maximum(slope, 0.0) if clip else slope)/L)
        if record:
            slope_hist[:, t], v_hist[:, t] = slope, v
    if excess:
        return mis, s
    if record:
        return mis, slope_hist, v_hist
    return mis


def fKF_batch(X, D, h, b_eta, v, chi_fn, flip_at=None, wait=None, excess=False):
    """B fKFs of eq. (37) side by side. Misalignment against h, or against -h from step flip_at on."""
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.asarray(v, dtype=float)
    mis = np.empty((B, n))
    if excess:
        s = np.empty((B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_{t-1})
        if t < first:
            continue
        power = np.einsum("bm,bm->b", x_t, x_t)
        chi, _ = chi_fn(e, np.sqrt(v*power), b_eta)
        w = w + x_t*(chi/power)[:, None]
    if excess:
        return mis, s
    return mis


# KF_batch: copied from vkf-kf.ipynb (commit 058a9f9); options flip_at and excess added (as in sKF_batch above),
# nothing else changed.
def KF_batch(X, D, h, b_eta, epsilon, chi_fn=chi_laplacian, flip_at=None, excess=False):
    L, (B, n) = len(h), X.shape
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    Sigma = np.broadcast_to(np.eye(L)*float(VAR_THETA_0), (B, L, L)).copy()
    eps_I = np.asarray(epsilon, dtype=float)[:, None, None]*np.eye(L)
    mis, k_hist = np.empty((B, n)), np.zeros((B, n))
    if excess:
        s = np.empty((B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_{t-1})
        if t < L:
            continue
        St = Sigma + eps_I
        Sx = np.einsum("bij,bj->bi", St, x_t)
        sigma2 = np.einsum("bi,bi->b", x_t, Sx)
        sigma = np.sqrt(sigma2)
        k_hist[:, t] = sigma/b_eta
        chi, slope = chi_fn(e, sigma, b_eta)
        w = w + Sx*(chi/sigma2)[:, None]
        Sigma = St - (slope/sigma2)[:, None, None]*np.einsum("bi,bj->bij", Sx, Sx)
        Sigma = 0.5*(Sigma + np.swapaxes(Sigma, 1, 2))
    if excess:
        return mis, k_hist, s
    return mis, k_hist


# === copied from bank-skf-fkf.ipynb section 4 (commit 57bd568); bank_batch has the option excess added ===
LOG_2PI = np.log(2*np.pi)


def transition_matrix(K, q):
    """p_ij of eq. (eps.chain): q to each neighbour on the grid, 1 - q n_i to stay."""
    P = np.zeros((K, K))
    for i in range(K):
        neighbours = [j for j in (i - 1, i + 1) if 0 <= j < K]   # n_i = 1 at the ends, 2 inside
        P[i, neighbours] = q
        P[i, i] = 1 - q*len(neighbours)
    return P


def log_density_laplacian(e, sigma, b_eta):
    """log p_t(e) of eq. (bank.weights), from eq. (NL.density): log phi(u) + log(R(k-u) + R(k+u)) - log 2b."""
    u = e/sigma                                        # u = e / sigma
    k_t = sigma/b_eta                                  # k = sigma / b_eta
    return (-0.5*u**2 - 0.5*LOG_2PI                    # log phi(u)
            + np.logaddexp(log_mills(k_t - u), log_mills(k_t + u))   # log(R(k - u) + R(k + u))
            - np.log(2*b_eta))


def bank_step(W, V, log_mu, x_t, d_t, power, grid, log_P, chi_fn, log_density, scale):
    """One step of Algorithm 1 (V given) or Algorithm 2 (V = None) for B banks side by side.
    W (K, B, L) locations, V (K, B) variances, log_mu (K, B) log weights, x_t (B, L), d_t and power (B,).
    Returns the new W, V, log_mu, and log c, log p and u = e/sigma of this step (K, B)."""
    L = W.shape[-1]
    # mixing, eqs. (bank.mixing.prob) and (bank.mixing)
    log_joint = log_P[:, :, None] + log_mu[:, None, :]   # log p_ij mu_{t-1,i}, (i, j, B)
    log_c = logsumexp(log_joint, axis=0)                 # log c_{t,j}
    mix = np.exp(log_joint - log_c[None])                # mu_{t-1,i|j}
    W_bar = np.einsum("ijb,ibm->jbm", mix, W)            # w_bar_{t,j}
    if V is not None:
        diff = W[:, None] - W_bar[None]                  # w_{t-1,i} - w_bar_{t,j}, (i, j, B, L)
        spread = np.einsum("ijbm,ijbm->ijb", diff, diff)/L
        V_bar = np.einsum("ijb,ijb->jb", mix, V[:, None] + spread)   # v_bar_{t,j}
        v_tilde = V_bar + grid[:, None]                  # v_tilde = v_bar + eps^(j)
    else:
        v_tilde = np.broadcast_to(grid[:, None], log_mu.shape)       # fKF: the fixed v^(j)
    # one sKF / fKF step per filter, eq. (bank.update)
    sigma = np.sqrt(v_tilde*power)                       # sigma_{t,j}
    e = d_t - np.einsum("jbm,bm->jb", W_bar, x_t)        # e_{t,j}, each filter's own error
    chi, slope = chi_fn(e, sigma, scale)
    W = W_bar + x_t*(chi/power)[..., None]
    if V is not None:
        V = v_tilde*(1 - slope/L)
    # weights, eq. (bank.weights), in logs
    log_p = log_density(e, sigma, scale)
    ell = log_c + log_p
    log_mu = ell - logsumexp(ell, axis=0)
    return W, V, log_mu, log_c, log_p, e/sigma


def bank_batch(X, D, h, grid, q, family, chi_fn, log_density, scale, v0=VAR_THETA_0, flip_at=None,
               wait=None, record=False, clamp=None, excess=False):
    """B banks side by side, one per row of X and D. Returns the misalignment of the output (B, n),
    the weights averaged over the rows (K, n) and the largest |u| = |e|/sigma met. record=True adds,
    per filter and step, log mu, log c, log p and the misalignment of each filter (K, B, n), and the
    output after each step (B, n, L). excess=True adds s_t = x_t'(theta_t - w_hat_{t-1}) (B, n)."""
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    grid = np.asarray(grid, dtype=float)
    K = len(grid)
    with np.errstate(divide="ignore"):
        log_P = np.log(transition_matrix(K, q))          # log 0 = -inf where p_ij = 0
    W = np.zeros((K, B, L))
    V = np.full((K, B), float(v0)) if family == "sKF" else None
    log_mu = np.full((K, B), -np.log(K))                 # mu_0 = 1/K
    x_t, w_hat = np.zeros((B, L)), np.zeros((B, L))
    mis = np.empty((B, n))
    if excess:
        s = np.empty((B, n))
    mu_mean = np.empty((K, n))
    u_max = 0.0
    if record:
        rec = {key: np.full((K, B, n), np.nan) for key in ("log_mu", "log_c", "log_p", "mis_j")}
        rec["w_hat"] = np.full((B, n, L), np.nan)
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        target = h if flip_at is None or t < flip_at else -h
        dw = w_hat - target
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_hat_{t-1})
        if record:
            dW = W - target
            rec["mis_j"][:, :, t] = np.einsum("jbm,jbm->jb", dW, dW)
        if t >= first:
            power = np.einsum("bm,bm->b", x_t, x_t)
            W, V, log_mu, log_c, log_p, u = bank_step(W, V, log_mu, x_t, D[:, t], power, grid, log_P,
                                                      chi_fn, log_density, scale)
            if clamp is not None:                        # Leszek's mu = max(mu, clamp), renormalized
                log_mu = np.maximum(log_mu - log_mu.max(axis=0), np.log(clamp))
                log_mu = log_mu - logsumexp(log_mu, axis=0)
            w_hat = np.einsum("jb,jbm->bm", np.exp(log_mu), W)          # eq. (bank.output)
            u_max = max(u_max, float(np.abs(u).max()))
            if record:
                rec["log_c"][:, :, t], rec["log_p"][:, :, t], rec["w_hat"][:, t] = log_c, log_p, w_hat
        mu_mean[:, t] = np.exp(log_mu).mean(axis=1)
        if record:
            rec["log_mu"][:, :, t] = log_mu
    out = {"mis": mis, "mu": mu_mean, "u_max": u_max}
    if excess:
        out["s"] = s
    if record:
        out.update(rec)
    return out


EPS_BANK = np.geomspace(1e-9, 1e-4, 6)     # the draft's remark, line 1614
V_BANK = np.geomspace(1e-6, 1e-2, 9)       # Leszek's fkfbank*.py
Q_BANK = 1e-4
# === end of the copied block ===

# Copied from bank-skf-fkf.ipynb section 4 (commit 57bd568), NOT edited.
def log_density_gaussian(e, sigma, v_eta):
    """log p_t(e) for Gaussian noise: N(0, sigma^2 + v_eta), eq. (bank.weights)."""
    s2 = sigma**2 + v_eta
    return -0.5*(LOG_2PI + np.log(s2)) - e**2/(2*s2)


print("filters ready")

### 3.1 The bank of KFs (Algorithm 3)

Each filter $j$ holds a location $\boldsymbol{w}_{t,j}$ and a full covariance $\boldsymbol{\Sigma}_{t,j}$. With the
mixing weights $\mu_{t-1,i|j} = p_{ij}\mu_{t-1,i}/c_{t,j}$ of eq. (53), the two equations that replace (54) and (55)
are

$$\bar{\boldsymbol{w}}_{t,j} = \sum_i \mu_{t-1,i|j}\,\boldsymbol{w}_{t-1,i}, \qquad
\bar{\boldsymbol{\Sigma}}_{t,j} = \sum_i \mu_{t-1,i|j}\Big(\boldsymbol{\Sigma}_{t-1,i}
+ (\boldsymbol{w}_{t-1,i} - \bar{\boldsymbol{w}}_{t,j})(\boldsymbol{w}_{t-1,i} - \bar{\boldsymbol{w}}_{t,j})^{\mathsf T}\Big),$$

$$\tilde{\boldsymbol{\Sigma}}_{t,j} = \bar{\boldsymbol{\Sigma}}_{t,j} + \varepsilon^{(j)}\boldsymbol{I}, \quad
\sigma_{t,j}^2 = \boldsymbol{x}_t^{\mathsf T}\tilde{\boldsymbol{\Sigma}}_{t,j}\boldsymbol{x}_t, \quad
e_{t,j} = y_t - \boldsymbol{x}_t^{\mathsf T}\bar{\boldsymbol{w}}_{t,j}, \quad
\boldsymbol{w}_{t,j} = \bar{\boldsymbol{w}}_{t,j} + \frac{\tilde{\boldsymbol{\Sigma}}_{t,j}\boldsymbol{x}_t}{\sigma_{t,j}^2}\,\chi_t(e_{t,j}), \quad
\boldsymbol{\Sigma}_{t,j} = \tilde{\boldsymbol{\Sigma}}_{t,j}
- \chi_t'(e_{t,j})\,\frac{\tilde{\boldsymbol{\Sigma}}_{t,j}\boldsymbol{x}_t\boldsymbol{x}_t^{\mathsf T}\tilde{\boldsymbol{\Sigma}}_{t,j}}{\sigma_{t,j}^2},$$

and the weights are eq. (56) as written, $\mu_{t,j} \propto c_{t,j}\,f_{t,j}(e_{t,j})$, with this $\sigma_{t,j}^2$ in place of
$\tilde{v}_{t,j}\lVert\boldsymbol{x}_t\rVert^2$. The output is $\hat{\boldsymbol{w}}_t = \sum_j \mu_{t,j}\boldsymbol{w}_{t,j}$.

- `_kf_mix_predict` and `_kf_update`: the covariance work, parallel over the $K \times B$ filters. Only the upper
  triangle of each covariance is computed or read; the symmetrisation that `KF_batch` does after each update is
  then implicit.
- `kf_bank_step` and `kf_bank_batch`: the same structure, options and outputs as `bank_step` and `bank_batch`.
- `kf_bank_reference`: Algorithm 3 line by line in plain numpy, with full covariances and the weights in the linear
  domain. For the checks only.
- Measured before writing the notebook (M1, 20 realisations, $M = 128$, $K = 6$): 2 ms a step, against 12 ms for
  the same step in plain numpy.

In [ ]:
# === new: the bank of KFs, Algorithm 3 (kf-bank-algorithm.tex) ===
# The O(K M^2) work (the mixing of the covariances, the prediction, Sigma~ x and the covariance update) runs in two
# numba kernels, parallel over the K x B filters; chi, chi' and the log density are the scipy functions above, called
# between the two kernels, so the Laplacian pieces are the same code as in every other filter here.
# Only the upper triangle (n >= m) of every covariance is computed or read. Sigma is symmetric, so the lower triangle
# is never needed, and the symmetrisation that KF_batch does after each update is implicit.
@nb.njit(parallel=True)
def _kf_mix_predict(W, S, mix, W_bar, grid, x, S_t, Sx, s2):
    """Eq. (KF.bank.mixing) for the covariances and the first line of eq. (KF.bank.update), K filters x B banks.
    W (K, B, L) and S (K, B, L, L) the filters after step t-1; mix (K, K, B) = mu_{t-1,i|j}; W_bar (K, B, L) the
    mixed locations; x (B, L). Writes S_t = Sigma~_{t,j} = Sigma_bar_{t,j} + eps^(j) I, Sx = Sigma~ x, s2 = x' Sigma~ x."""
    K, B, L = W.shape
    for bj in nb.prange(B*K):
        b, j = bj//K, bj % K                     # bank-major order: S[i, b] is reused by j - 1, j and j + 1
        T = S_t[j, b]
        for m in range(L):
            for n in range(m, L):
                T[m, n] = 0.0
        for i in range(K):
            c = mix[i, j, b]
            if c == 0.0:                         # p_ij = 0: i is not a neighbour of j
                continue
            Si = S[i, b]
            for m in range(L):
                dm = c*(W[i, b, m] - W_bar[j, b, m])
                for n in range(m, L):            # mu_{i|j} (Sigma_i + (w_i - w_bar_j)(w_i - w_bar_j)')
                    T[m, n] += c*Si[m, n] + dm*(W[i, b, n] - W_bar[j, b, n])
        for m in range(L):
            T[m, m] += grid[j]                   # + eps^(j) I
            Sx[j, b, m] = 0.0
        for m in range(L):                       # Sigma~ x from the upper triangle
            xm = x[b, m]
            acc = T[m, m]*xm
            for n in range(m + 1, L):
                acc += T[m, n]*x[b, n]
                Sx[j, b, n] += T[m, n]*xm
            Sx[j, b, m] += acc
        tot = 0.0
        for m in range(L):
            tot += Sx[j, b, m]*x[b, m]
        s2[j, b] = tot                           # sigma_{t,j}^2 = x' Sigma~ x


@nb.njit(parallel=True)
def _kf_update(S_t, Sx, coef, S):
    """Second line of eq. (KF.bank.update), upper triangle: S = Sigma~ - coef (Sigma~ x)(Sigma~ x)', coef = chi'/sigma^2."""
    K, B, L = Sx.shape
    for bj in nb.prange(B*K):
        b, j = bj//K, bj % K
        c = coef[j, b]
        T, U = S_t[j, b], S[j, b]
        for m in range(L):
            cm = c*Sx[j, b, m]
            for n in range(m, L):
                U[m, n] = T[m, n] - cm*Sx[j, b, n]


def full_cov(S):
    """The symmetric matrices that the upper triangles S (..., L, L) stand for."""
    return np.triu(S) + np.swapaxes(np.triu(S, 1), -1, -2)


def kf_bank_step(W, S, log_mu, x_t, d_t, grid, log_P, chi_fn, log_density, scale, work):
    """One step of Algorithm 3 for B banks side by side, as bank_step does for Algorithms 1 and 2.
    W (K, B, L) locations, S (K, B, L, L) covariances (upper triangles, updated in place), log_mu (K, B), x_t (B, L),
    d_t (B,), work = (S_t, Sx, s2) buffers of shapes (K, B, L, L), (K, B, L), (K, B).
    Returns the new W and log_mu, and log c, log p and u = e/sigma of this step (K, B)."""
    # mixing, eqs. (bank.mixing.prob) and (KF.bank.mixing)
    log_joint = log_P[:, :, None] + log_mu[:, None, :]   # log p_ij mu_{t-1,i}, (i, j, B)
    log_c = logsumexp(log_joint, axis=0)                 # log c_{t,j}
    mix = np.exp(log_joint - log_c[None])                # mu_{t-1,i|j}
    W_bar = np.einsum("ijb,ibm->jbm", mix, W)            # w_bar_{t,j}
    S_t, Sx, s2 = work
    _kf_mix_predict(W, S, mix, W_bar, grid, x_t, S_t, Sx, s2)   # Sigma_bar_{t,j} + eps^(j) I, Sigma~ x, sigma^2
    # one KF step per filter, eq. (KF.bank.update)
    sigma = np.sqrt(s2)                                  # sigma_{t,j}
    e = d_t - np.einsum("jbm,bm->jb", W_bar, x_t)        # e_{t,j}, each filter's own error
    chi, slope = chi_fn(e, sigma, scale)
    W = W_bar + Sx*(chi/s2)[..., None]                   # w_{t,j}
    _kf_update(S_t, Sx, slope/s2, S)                     # Sigma_{t,j}
    # weights, eq. (bank.weights) with sigma^2 = x' Sigma~ x, in logs
    log_p = log_density(e, sigma, scale)
    ell = log_c + log_p
    log_mu = ell - logsumexp(ell, axis=0)
    return W, log_mu, log_c, log_p, e/sigma


def kf_bank_batch(X, D, h, grid, q, chi_fn, log_density, scale, v0=VAR_THETA_0, flip_at=None, wait=None,
                  record=False, clamp=None, excess=False):
    """B banks of KFs side by side (Algorithm 3), one per row of X and D, with the options and the outputs of
    bank_batch. record=True also returns the covariances after the last step (K, B, L, L), upper triangles."""
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    grid = np.asarray(grid, dtype=float)
    K = len(grid)
    with np.errstate(divide="ignore"):
        log_P = np.log(transition_matrix(K, q))          # log 0 = -inf where p_ij = 0
    W = np.zeros((K, B, L))
    S = np.zeros((K, B, L, L))
    S[:, :, np.arange(L), np.arange(L)] = float(v0)     # Sigma_0 = v0 I
    work = (np.zeros((K, B, L, L)), np.zeros((K, B, L)), np.zeros((K, B)))
    log_mu = np.full((K, B), -np.log(K))                 # mu_0 = 1/K
    x_t, w_hat = np.zeros((B, L)), np.zeros((B, L))
    mis = np.empty((B, n))
    if excess:
        s = np.empty((B, n))
    mu_mean = np.empty((K, n))
    u_max = 0.0
    if record:
        rec = {key: np.full((K, B, n), np.nan) for key in ("log_mu", "log_c", "log_p", "mis_j")}
        rec["w_hat"] = np.full((B, n, L), np.nan)
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        target = h if flip_at is None or t < flip_at else -h
        dw = w_hat - target
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_hat_{t-1})
        if record:
            dW = W - target
            rec["mis_j"][:, :, t] = np.einsum("jbm,jbm->jb", dW, dW)
        if t >= first:
            W, log_mu, log_c, log_p, u = kf_bank_step(W, S, log_mu, x_t, D[:, t], grid, log_P, chi_fn, log_density,
                                                      scale, work)
            if clamp is not None:                        # Leszek's mu = max(mu, clamp), renormalized
                log_mu = np.maximum(log_mu - log_mu.max(axis=0), np.log(clamp))
                log_mu = log_mu - logsumexp(log_mu, axis=0)
            w_hat = np.einsum("jb,jbm->bm", np.exp(log_mu), W)          # eq. (bank.output)
            u_max = max(u_max, float(np.abs(u).max()))
            if record:
                rec["log_c"][:, :, t], rec["log_p"][:, :, t], rec["w_hat"][:, t] = log_c, log_p, w_hat
        mu_mean[:, t] = np.exp(log_mu).mean(axis=1)
        if record:
            rec["log_mu"][:, :, t] = log_mu
    out = {"mis": mis, "mu": mu_mean, "u_max": u_max}
    if excess:
        out["s"] = s
    if record:
        out.update(rec)
        out["W"], out["S"] = W, S
    return out


def kf_bank_reference(X, D, L, grid, q, chi_fn, log_density, scale, v0=VAR_THETA_0, wait=None):
    """Algorithm 3 line by line in plain numpy, for the checks only: full covariances, the weights in the linear
    domain, loops over j and i. Returns the output after every step (B, n, L) and log mu (K, B, n)."""
    B, n = X.shape
    K = len(grid)
    first = L if wait is None else wait
    P = transition_matrix(K, q)
    W = np.zeros((K, B, L))                                          # line 1
    S = np.broadcast_to(float(v0)*np.eye(L), (K, B, L, L)).copy()
    mu = np.full((K, B), 1.0/K)
    x_t, w_hat = np.zeros((B, L)), np.zeros((B, L))
    w_hist, log_mu_hist = np.zeros((B, n, L)), np.zeros((K, B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)                                 # line 3
        if t >= first:
            c = np.einsum("ij,ib->jb", P, mu)                        # line 5: c_{t,j}
            mix = P[:, :, None]*mu[:, None, :]/c[None]               #         mu_{t-1,i|j}
            W_bar = np.einsum("ijb,ibm->jbm", mix, W)                # line 6
            S_bar = np.zeros((K, B, L, L))
            for j in range(K):                                       # line 7
                for i in range(K):
                    d = W[i] - W_bar[j]
                    S_bar[j] += mix[i, j][:, None, None]*(S[i] + d[:, :, None]*d[:, None, :])
            ell = np.empty((K, B))
            for j in range(K):
                S_tilde = S_bar[j] + grid[j]*np.eye(L)               # line 9
                sigma2 = np.einsum("bm,bmn,bn->b", x_t, S_tilde, x_t)
                e = D[:, t] - np.einsum("bm,bm->b", x_t, W_bar[j])
                chi, slope = chi_fn(e, np.sqrt(sigma2), scale)       # line 10
                g = np.einsum("bmn,bn->bm", S_tilde, x_t)/sigma2[:, None]
                W[j] = W_bar[j] + g*chi[:, None]                     # line 11
                S[j] = S_tilde - (slope*sigma2)[:, None, None]*g[:, :, None]*g[:, None, :]
                ell[j] = np.log(c[j]) + log_density(e, np.sqrt(sigma2), scale)   # line 12
            mu = np.exp(ell - ell.max(axis=0))                       # line 14
            mu = mu/mu.sum(axis=0)
            w_hat = np.einsum("jb,jbm->bm", mu, W)                   # line 15
        w_hist[:, t] = w_hat
        log_mu_hist[:, :, t] = np.log(mu)
    return w_hist, log_mu_hist

## 4. Checks of the bank of KFs

All must pass before any result. 4.1-4.3 compare it with code already checked; 4.4-4.6 with things written without
reference to Algorithm 3; 4.7 with Leszek's own bank of full KFs.

### 4.1 The fast code against Algorithm 3 in plain numpy

Four realisations of AR($-0.9$), 700 steps, the flip at step 400, both likelihoods. With the Gaussian $\chi$ the two
must agree to rounding. With the Laplacian $\chi$ rounding is amplified: at the first updates
$k_t = \sigma_t/b_\eta \approx 125$, where `log_mills` subtracts two terms of about $k_t^2/2 \approx 7800$, so one bit of
difference in $\sigma_t$ (the kernels add in another order than `einsum`) moves $\chi_t$ by about $10^{-9}$, and the
recursion carries it on. The yardstick printed beside: the reference against itself with $\boldsymbol{x}_t$ scaled by
$1 + 2^{-52}$, one unit in the last place.

In [ ]:
start = time.time()
X_c41, D_c41 = stack([signals(seed, -0.9, 700, change_at=400) for seed in range(4)])
LIK_CHECK = {"Laplacian": (chi_laplacian, log_density_laplacian, noise_of(-0.9)[1]),
             "Gaussian": (chi_gaussian, log_density_gaussian, noise_variance(-0.9))}


def gaps(w_a, log_mu_a, w_b, log_mu_b):
    # output: largest ||w_a - w_b|| / ||w_b||; weights: largest |log mu_a - log mu_b| where mu_b > 1e-6
    w_gap = np.max(np.linalg.norm(w_a - w_b, axis=2)[:, M:]/np.linalg.norm(w_b, axis=2)[:, M:])
    big = np.exp(log_mu_b[:, :, M:]) > 1e-6
    return w_gap, np.max(np.abs(log_mu_a[:, :, M:] - log_mu_b[:, :, M:])[big])


print(f"{'':<17}{'fast against reference':-^30}  {'reference against itself, one ulp':-^36}")
print(f"{'likelihood':<11}{'q':>6}{'output':>15}{'log mu':>15}  {'output':>18}{'log mu':>18}")
for name, lik in LIK_CHECK.items():
    for q in (1e-4, 0.2):
        ref_w, ref_log_mu = kf_bank_reference(X_c41, D_c41, M, EPS_BANK, q, *lik)
        ulp_w, ulp_log_mu = kf_bank_reference(X_c41*(1 + 2.0**-52), D_c41, M, EPS_BANK, q, *lik)
        out = kf_bank_batch(X_c41, D_c41, ho, EPS_BANK, q, *lik, flip_at=400, record=True)
        fast = gaps(np.nan_to_num(out["w_hat"], nan=0.0), out["log_mu"], ref_w, ref_log_mu)
        ulp = gaps(ulp_w, ulp_log_mu, ref_w, ref_log_mu)
        print(f"{name:<11}{q:>6g}{fast[0]:>15.1e}{fast[1]:>15.1e}  {ulp[0]:>18.1e}{ulp[1]:>18.1e}")
print("   output: largest ||w_hat_t - w_hat_t(reference)|| / ||w_hat_t(reference)||; log mu: largest |difference| "
      "where mu > 1e-6")
del X_c41, D_c41, out
print(f"[{time.time() - start:.0f} s]")

### 4.2 One filter ($K = 1$) against `KF_batch`

A bank with one value of $\varepsilon$ is the KF of `vkf-kf.ipynb` (checked there against a textbook Kalman filter),
across a flip. Four realisations, 3000 steps, the flip at 1500.

In [ ]:
X_c42, D_c42 = stack([signals(seed, -0.9, 3000, change_at=1500) for seed in range(4)])
b_c42 = noise_of(-0.9)[1]
print(f"{'eps':>9}{'misalignment':>16}{'s_t':>12}{'KF_batch against itself, one ulp':>36}")
for eps in (1e-7, 1e-6, 1e-5):
    mis_kf, _, s_kf = KF_batch(X_c42, D_c42, ho, b_c42, np.full(4, eps), flip_at=1500, excess=True)
    mis_ulp, _ = KF_batch(X_c42*(1 + 2.0**-52), D_c42, ho, b_c42, np.full(4, eps), flip_at=1500)
    out = kf_bank_batch(X_c42, D_c42, ho, [eps], 0.0, chi_laplacian, log_density_laplacian, b_c42, flip_at=1500,
                        excess=True)
    print(f"{eps:>9.0e}{np.max(np.abs(out['mis'] - mis_kf)/mis_kf):>16.1e}"
          f"{np.max(np.abs(out['s'] - s_kf))/np.max(np.abs(s_kf)):>12.1e}{np.max(np.abs(mis_ulp - mis_kf)/mis_kf):>36.1e}")
print("   misalignment: largest relative difference; s_t: largest |difference| / largest |s_t|")
del X_c42, D_c42

### 4.3 $q = 0$

The filters no longer exchange anything: each must be `KF_batch` at its own $\varepsilon$, and the weights must be
eq. (58), $\mu_{T,j} \propto \mu_{0,j}\prod_t f_{t,j}(e_{t,j})$.

In [ ]:
X_c43, D_c43 = stack([signals(seed, -0.9, 2000, change_at=1000) for seed in range(4)])
b_c43 = noise_of(-0.9)[1]
out = kf_bank_batch(X_c43, D_c43, ho, EPS_BANK, 0.0, chi_laplacian, log_density_laplacian, b_c43, flip_at=1000,
                    record=True)
worst = worst_ulp = 0.0
for j, eps in enumerate(EPS_BANK):
    mis_kf, _ = KF_batch(X_c43, D_c43, ho, b_c43, np.full(4, eps), flip_at=1000)
    mis_ulp, _ = KF_batch(X_c43*(1 + 2.0**-52), D_c43, ho, b_c43, np.full(4, eps), flip_at=1000)
    worst = max(worst, np.max(np.abs(out["mis_j"][j] - mis_kf)/mis_kf))
    worst_ulp = max(worst_ulp, np.max(np.abs(mis_ulp - mis_kf)/mis_kf))
# eq. (eps.likelihood): log mu_t,j = log mu_0,j + sum of log p over the steps so far, normalised
log_lik = np.nancumsum(out["log_p"], axis=2) - np.log(len(EPS_BANK))
log_mu_58 = log_lik - logsumexp(log_lik, axis=0)
print(f"each filter of the bank against KF_batch at its eps: misalignment, largest relative difference {worst:.1e} "
      f"(KF_batch against itself, one ulp: {worst_ulp:.1e})")
print(f"weights against eq. (eps.likelihood): largest |difference| of log mu {np.max(np.abs(log_mu_58 - out['log_mu'])[:, :, M:]):.1e}")
del X_c43, D_c43, out

### 4.4 From a scalar prior: the first update and the second mixing are the sKF bank's

From $\boldsymbol{\Sigma}_{0,j} = v_0\boldsymbol{I}$, the first KF step is the sKF step, so the locations and the
weights must be the sKF bank's, and $\mathrm{tr}(\boldsymbol{\Sigma}_{1,j})/M = v_{1,j}$. At the second step the
mixed covariance is no longer scalar, but its trace over $M$ must be $\bar{v}_{2,j}$ of eq. (54) computed from the sKF
bank's state ($q = 0.2$, so that the mixing does mix). This checks the spread term of the mixing.

In [ ]:
# ---------------- 4.4: from a scalar prior, the first update and the second mixing are the sKF bank's ----------------
def check_scalar_start():
    X_r, D_r = stack([signals(seed, -0.9, M + 2) for seed in range(4)])
    lik = (chi_laplacian, log_density_laplacian, noise_of(-0.9)[1])
    K, q = len(EPS_BANK), 0.2                             # a large q, so the second mixing does mix
    with np.errstate(divide="ignore"):
        log_P = np.log(transition_matrix(K, q))
    W_s, V_s, log_mu_s = np.zeros((K, 4, M)), np.full((K, 4), VAR_THETA_0), np.full((K, 4), -np.log(K))
    W_k, log_mu_k = np.zeros((K, 4, M)), np.full((K, 4), -np.log(K))
    S_k = np.zeros((K, 4, M, M))
    S_k[:, :, np.arange(M), np.arange(M)] = VAR_THETA_0
    work = (np.zeros((K, 4, M, M)), np.zeros((K, 4, M)), np.zeros((K, 4)))
    x_t = np.zeros((4, M))
    for t in range(M):
        x_t = _roll_in(X_r[:, t], x_t)
    for step, t in enumerate((M, M + 1)):
        x_t = _roll_in(X_r[:, t], x_t)
        power = np.einsum("bm,bm->b", x_t, x_t)
        if step == 1:                                     # v_bar of eq. (bank.mixing) from the sKF bank's state
            log_joint = log_P[:, :, None] + log_mu_s[:, None, :]
            mix = np.exp(log_joint - logsumexp(log_joint, axis=0)[None])
            W_bar = np.einsum("ijb,ibm->jbm", mix, W_s)
            diff = W_s[:, None] - W_bar[None]
            V_bar = np.einsum("ijb,ijb->jb", mix, V_s[:, None] + np.einsum("ijbm,ijbm->ijb", diff, diff)/M)
        W_s, V_s, log_mu_s, *_ = bank_step(W_s, V_s, log_mu_s, x_t, D_r[:, t], power, EPS_BANK, log_P, *lik)
        W_k, log_mu_k, *_ = kf_bank_step(W_k, S_k, log_mu_k, x_t, D_r[:, t], EPS_BANK, log_P, *lik, work)
        if step == 0:
            tr = np.trace(full_cov(S_k), axis1=2, axis2=3)/M
            print(f"first update: locations max rel. diff {np.max(np.abs(W_k - W_s))/np.max(np.abs(W_s)):.1e}, "
                  f"log mu max |diff| {np.max(np.abs(log_mu_k - log_mu_s)):.1e}, "
                  f"tr(Sigma)/M against v max rel. diff {np.max(np.abs(tr - V_s)/V_s):.1e}")
        else:
            tr_bar = np.trace(full_cov(work[0]), axis1=2, axis2=3)/M - EPS_BANK[:, None]   # Sigma~ - eps I
            print(f"second mixing: tr(Sigma_bar)/M against v_bar of eq. (bank.mixing), max rel. diff "
                  f"{np.max(np.abs(tr_bar - V_bar)/V_bar):.1e}")


check_scalar_start()

### 4.5 The Gaussian $\chi$: a textbook IMM written without reference to Algorithm 3

With Gaussian noise the bank of KFs is the interacting multiple model estimator of tracking for $K$ random-walk
models. `imm_textbook` is written from that algorithm's cycle (mixing probabilities, mixed initial conditions,
mode-matched Kalman filters with the gain $\boldsymbol{P}\boldsymbol{x}_t/S$, mode probabilities, combination), not
from Algorithm 3. White input, $M = 16$, three values of $\varepsilon$, a flip at step 300, three realisations.

In [ ]:
# ---------------- 4.5: with the Gaussian chi, a textbook IMM written without reference to Algorithm 3 ----------------
def imm_textbook(x, d, L, grid, q, v_eta, v0):
    """The interacting multiple model estimator for one realisation, written from its standard cycle (mixing
    probabilities, mixed initial conditions, mode-matched Kalman filters, mode probabilities, combination), for K
    random-walk models theta_t = theta_{t-1} + xi_t, Cov(xi_t) = eps_j I, observed through y_t = x_t' theta_t + eta_t,
    eta_t ~ N(0, v_eta). The mode likelihoods are kept in logs. Returns the combined estimate after every step and
    the mode probabilities."""
    K, n = len(grid), len(x)
    P = transition_matrix(K, q)                           # P[i, j] = Pr(mode j at t | mode i at t-1)
    est, cov = [np.zeros(L) for _ in range(K)], [v0*np.eye(L) for _ in range(K)]
    prob = np.full(K, 1.0/K)
    reg = np.zeros(L)
    combined, probs = np.zeros((n, L)), np.zeros((n, K))
    for t in range(n):
        reg = np.concatenate([[x[t]], reg[:-1]])          # (x_t, x_{t-1}, ..., x_{t-L+1})
        if t >= L:
            c_bar = P.T @ prob                            # predicted mode probabilities
            mix = P*prob[:, None]/c_bar[None, :]          # mixing probabilities mu_{i|j}
            log_lik, new_est, new_cov = np.zeros(K), [], []
            for j in range(K):
                x0 = sum(mix[i, j]*est[i] for i in range(K))                          # mixed initial condition
                P0 = sum(mix[i, j]*(cov[i] + np.outer(est[i] - x0, est[i] - x0)) for i in range(K))
                P_pred = P0 + grid[j]*np.eye(L)           # prediction: F = I, Q = eps_j I
                S_inn = reg @ P_pred @ reg + v_eta        # innovation variance, H = x_t'
                gain = P_pred @ reg/S_inn                 # Kalman gain
                innov = d[t] - reg @ x0
                new_est.append(x0 + gain*innov)
                new_cov.append((np.eye(L) - np.outer(gain, reg)) @ P_pred)
                log_lik[j] = -0.5*innov**2/S_inn - 0.5*np.log(2*np.pi*S_inn)
            est, cov = new_est, new_cov
            a = log_lik + np.log(c_bar)
            prob = np.exp(a - a.max())/np.sum(np.exp(a - a.max()))                   # mode probabilities
        combined[t] = sum(prob[j]*est[j] for j in range(K))
        probs[t] = prob
    return combined, probs


def check_imm():
    rng = np.random.default_rng(7)
    L, n, flip, v_eta, v0 = 16, 600, 300, 0.01, 0.1
    grid = np.array([1e-6, 1e-4, 1e-2])
    h16 = rng.standard_normal(L)
    h16 = h16/np.linalg.norm(h16)
    print(f"{'q':>7}{'estimate, max rel. diff':>26}{'mode probabilities, max |diff|':>33}")
    for q in (1e-4, 1e-2, 0.3):
        worst_w = worst_p = 0.0
        for r in range(3):
            x = rng.standard_normal(n)
            y = np.convolve(h16, x)[:n]
            y[flip:] = -y[flip:]
            d = y + np.sqrt(v_eta)*rng.standard_normal(n)
            ref_w, ref_p = imm_textbook(x, d, L, grid, q, v_eta, v0)
            out = kf_bank_batch(x[None], d[None], h16, grid, q, chi_gaussian, log_density_gaussian, v_eta, v0=v0,
                                record=True)
            w = out["w_hat"][0, L:]
            worst_w = max(worst_w, np.max(np.linalg.norm(w - ref_w[L:], axis=1)/np.linalg.norm(ref_w[L:], axis=1)))
            worst_p = max(worst_p, np.max(np.abs(np.exp(out["log_mu"][:, 0, L:]).T - ref_p[L:])))
        print(f"{q:>7g}{worst_w:>26.1e}{worst_p:>33.1e}")


check_imm()

### 4.6 The brute-force mixture over $\varepsilon$ sequences

Check 5 of the bank notebook with the KF step: one filter per sequence $\varepsilon_1, \dots, \varepsilon_t$, no merge.
The bank must match it at the first update and, for $q = 0$, at every update; otherwise the difference is what the
merge costs. The small problem of the bank notebook ($M = 4$, $K = 3$, 12 updates, the flip after the 6th).

In [ ]:
# ---------------- 4.6: the brute-force mixture over eps sequences ----------------
# enumerate_sequences: copied from bank-skf-fkf.ipynb section 5.5 (commit 57bd568); one family added, "KF", in which
# every sequence carries its own covariance and takes the KF step of eqs. (KF.mean)-(KF.cov). Nothing else changed.
def enumerate_sequences(x, d, grid, q, family, chi_fn, log_density, scale, v0):
    """One filter per eps sequence, no merging. Returns Pr(eps_t = j | y_1:t) (n, K), the mixture mean of
    theta (n, L) and the number of sequences at each step."""
    L, K, n = M_SMALL, len(grid), len(x)
    P = transition_matrix(K, q)
    W, V = np.zeros((K, L)), np.full(K, float(v0))
    if family == "KF":
        S = np.broadcast_to(float(v0)*np.eye(L), (K, L, L)).copy()
    log_w, last = np.full(K, -np.log(K)), np.arange(K)     # eps_0 = j with probability 1/K
    x_t = np.zeros(L)
    probs, means, count = np.full((n, K), np.nan), np.full((n, L), np.nan), np.zeros(n, dtype=int)
    for t in range(n):
        x_t = np.roll(x_t, 1)
        x_t[0] = x[t]
        if t < L:
            continue
        parent, child = np.nonzero(P[last] > 0)            # every move i -> j of positive probability
        log_w = log_w[parent] + np.log(P[last[parent], child])
        W, V, last = W[parent], V[parent], child
        if family == "KF":
            S_tilde = S[parent] + grid[last][:, None, None]*np.eye(L)
            Sx = S_tilde @ x_t
            sigma = np.sqrt(Sx @ x_t)
            e = d[t] - W @ x_t
            chi, slope = chi_fn(e, sigma, scale)
            W = W + Sx*(chi/sigma**2)[:, None]
            S = S_tilde - (slope/sigma**2)[:, None, None]*Sx[:, :, None]*Sx[:, None, :]
        else:
            power = x_t @ x_t
            v_tilde = V + grid[last] if family == "sKF" else grid[last]
            sigma = np.sqrt(v_tilde*power)
            e = d[t] - W @ x_t
            chi, slope = chi_fn(e, sigma, scale)
            W = W + np.outer(chi/power, x_t)
            if family == "sKF":
                V = v_tilde*(1 - slope/L)
        log_w = log_w + log_density(e, sigma, scale)
        log_w = log_w - logsumexp(log_w)
        probs[t] = np.bincount(last, weights=np.exp(log_w), minlength=K)
        means[t] = np.exp(log_w) @ W
        count[t] = len(log_w)
    return probs, means, count


M_SMALL, UPDATES, B_SMALL, V0_SMALL = 4, 12, 0.1, 0.01


def check_brute_force():
    # the small problem of the bank notebook's check 5, drawn as there
    rng_small = np.random.default_rng(5)
    h_small = rng_small.standard_normal(M_SMALL)
    h_small = h_small/np.linalg.norm(h_small)
    n_small = M_SMALL + UPDATES
    x_small = rng_small.standard_normal(n_small)
    y_small = np.convolve(h_small, x_small)[:n_small]
    y_small[M_SMALL + 6:] = -y_small[M_SMALL + 6:]
    d_small = y_small + rng_small.laplace(0, B_SMALL, n_small)
    grid = np.array([1e-4, 1e-3, 1e-2])
    print(f"{'':<15}{'q':>7}{'1st update':>12}{'2nd':>10}{'largest':>10}{'output, largest':>18}{'sequences':>11}")
    for name, (chi_fn, log_density, scale) in (("KF, Laplacian", (chi_laplacian, log_density_laplacian, B_SMALL)),
                                               ("KF, Gaussian", (chi_gaussian, log_density_gaussian, 2*B_SMALL**2))):
        for q in (0.0, 1e-4, 1e-2, 0.2):
            probs, means, count = enumerate_sequences(x_small, d_small, grid, q, "KF", chi_fn, log_density, scale,
                                                      V0_SMALL)
            out = kf_bank_batch(x_small[None], d_small[None], h_small, grid, q, chi_fn, log_density, scale,
                                v0=V0_SMALL, record=True)
            mu = np.exp(out["log_mu"][:, 0, M_SMALL:]).T
            gap = np.abs(mu - probs[M_SMALL:]).max(axis=1)
            w_gap = (np.linalg.norm(out["w_hat"][0, M_SMALL:] - means[M_SMALL:], axis=1)
                     / np.linalg.norm(means[M_SMALL:], axis=1))
            print(f"{name:<15}{q:>7g}{gap[0]:>12.1e}{gap[1]:>10.1e}{gap.max():>10.1e}{w_gap.max():>18.1e}"
                  f"{count[-1]:>11d}")
    print("   1st update / 2nd / largest: max over j of |mu_t,j - Pr(eps_t = j | y_1:t)|, over the 12 updates")
    print("   output: largest ||w_hat_t - mixture mean_t|| / ||mixture mean_t||")


check_brute_force()

### 4.7 Anchor: Leszek's bank of full KFs

`sims/driftkf.py` (Overleaf commit `fc54c65`) runs a bank of full KFs with the same mixing; it is behind the row
"coloured, full KF" of the draft's preliminary-results remark: best fixed $\varepsilon$ $-8.2$ dB, bank $-10.7$ dB.
His signals and his reading, written again line for line around `kf_bank_step`, as the bank notebook's section 6 did
for `driftbench.py`. His log is not in the Overleaf folder, so the numbers to match are the draft's, to one decimal.

In [ ]:
# ---------------- 4.7: anchor, Leszek's bank of full KFs ----------------
# Written from Leszek's sims/driftkf.py (Overleaf commit fc54c65), line for line, as the bank notebook's section 6 did
# for driftbench.py: AR = 0.9 in his sign (x_t = -0.9 x_{t-1} + u_t, our AR(-0.9)), beta* = 0.2, R = 20, 5 dB with
# E[(x'h)^2] taken as 1, b_eta = E|eta|, his random response with a 0.2 s decay; the response drifts with variance
# 1e-8, 1e-7, 1e-6, 1e-8 in four segments of 6000 samples and changes sign at the three boundaries. His regressor is a
# full window from t = 0, so he updates from the first step; v0 = 1/M; misalignment after the update, divided by
# ||H_t||^2; the score is the mean over the record in dB. A fixed eps is the bank with K = 1.
def leszek_kf_signals():
    R_K, SEG_K, EPS_TRUE_K, AR_K = 20, 6000, [1e-8, 1e-7, 1e-6, 1e-8], 0.9
    T_K = SEG_K*len(EPS_TRUE_K)
    rng0 = np.random.default_rng(3)
    u = rng0.standard_normal((R_K, T_K + M))
    x = np.empty_like(u)
    x[:, 0] = u[:, 0]
    for i in range(1, T_K + M):
        x[:, i] = -AR_K*x[:, i - 1] + u[:, i]
    x *= np.sqrt(1 - AR_K**2)
    veta = 10**(-SNR_DB/10)
    alpha = np.sqrt(veta*G(1/BETA)/G(3/BETA))            # gg_alpha of his ggq.py
    eta = np.sign(rng0.standard_normal((R_K, T_K)))*alpha*rng0.gamma(1/BETA, 1, (R_K, T_K))**(1/BETA)
    b = np.sqrt(veta)*G(2/BETA)/np.sqrt(G(1/BETA)*G(3/BETA))
    h0 = rng0.standard_normal(M)*np.exp(-6.9*np.arange(M)/(0.2*8000))
    h0 = h0/np.linalg.norm(h0)

    def system():
        # his system(): yields H_t (R, M); the same path in every call
        r = np.random.default_rng(11)
        H = np.tile(h0, (R_K, 1))
        for t in range(T_K):
            if t > 0 and t % SEG_K == 0:
                H = -H                                   # abrupt change: sign flip
            H = H + np.sqrt(EPS_TRUE_K[t//SEG_K])*r.standard_normal((R_K, M))
            yield H
    return x, eta, b, system


def run_leszek_kf(sig, grid, q, clamp=None):
    # his run_bank (and run_single, K = 1), through kf_bank_step
    x, eta, b, system = sig
    grid = np.asarray(grid, dtype=float)
    K, R_K = len(grid), len(x)
    with np.errstate(divide="ignore"):
        log_P = np.log(transition_matrix(K, q))
    W, log_mu = np.zeros((K, R_K, M)), np.full((K, R_K), -np.log(K))
    S = np.zeros((K, R_K, M, M))
    S[:, :, np.arange(M), np.arange(M)] = 1.0/M          # his S = I/M
    work = (np.zeros((K, R_K, M, M)), np.zeros((K, R_K, M)), np.zeros((K, R_K)))
    mis = []
    for t, H in enumerate(system()):
        X = np.ascontiguousarray(x[:, t:t + M][:, ::-1])
        y = np.einsum("rm,rm->r", X, H) + eta[:, t]
        W, log_mu, *_ = kf_bank_step(W, S, log_mu, X, y, grid, log_P, chi_laplacian, log_density_laplacian, b, work)
        if clamp is not None:
            log_mu = np.maximum(log_mu - log_mu.max(axis=0), np.log(clamp))
            log_mu = log_mu - logsumexp(log_mu, axis=0)
        w = np.einsum("jr,jrm->rm", np.exp(log_mu), W)
        mis.append(np.mean(np.sum((w - H)**2, axis=1)/np.sum(H**2, axis=1)))
    return 10*np.log10(np.mean(mis))


start = time.time()
SIG_K = leszek_kf_signals()
fixed = {eps: run_leszek_kf(SIG_K, [eps], 0.0) for eps in np.geomspace(1e-9, 1e-4, 6)}
print("fixed eps (bank with K = 1): " + ", ".join(f"{eps:.0e} {m:.2f}" for eps, m in fixed.items()))
best = min(fixed, key=fixed.get)
anchor_bank = {clamp: run_leszek_kf(SIG_K, EPS_BANK, 1e-4, clamp=clamp) for clamp in (1e-200, None)}
print(f"\n{'run':<40}{'ours [dB]':>10}   Leszek (draft, preliminary-results remark)")
print(f"{f'best fixed eps ({best:.0e})':<40}{fixed[best]:>10.2f}   -8.2")
print(f"{'bank of KFs, q = 1e-4, clamp 1e-200':<40}{anchor_bank[1e-200]:>10.2f}   -10.7")
print(f"{'bank of KFs, q = 1e-4, no clamp':<40}{anchor_bank[None]:>10.2f}")
del SIG_K
print(f"[{time.time() - start:.0f} s]")

## 5. Measurement

Section 5 of `bank-kf-excess-error.ipynb` (commit `f2ded06`), nothing edited: notebook 09's `steady_state`,
`is_settled`, `recovery`, and `time_to`, `first_below`, `measure`, `pick`, the KF search `sweep_kf`, and the excess
error `smoothed_ratio`, `measure_excess`.

In [ ]:
# === copied from bank-kf-excess-error.ipynb section 5 (commit f2ded06), NOT edited ===
# === IGNACIO: steady_state, is_settled and recovery - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/09_matcheado.ipynb, commit 6e6dabc,
#         via bank-skf-fkf.ipynb (commit 57bd568)
def steady_state(misalignment):
    """Floor in dB, and the first step within 3 dB of it."""
    tail = slice(3*len(misalignment)//4, len(misalignment))
    floor = 10*np.log10(misalignment[tail].mean())
    db = 10*np.log10(misalignment)
    reached = int(np.argmax(db < floor + 3))
    return floor, reached


def is_settled(misalignment, drift):
    """Settled within the run: within 3 dB of the floor by half of it, and no longer descending."""
    n = len(misalignment)
    floor, reached = steady_state(misalignment)
    third_quarter = 10*np.log10(misalignment[n//2:3*n//4].mean())
    # reached = 0: the run never left the 3 dB band around its start, so it never converged either
    return bool(0 < reached <= n/2 and third_quarter - floor <= drift)


def recovery(misalignment, floor):
    """Steps from the change until the misalignment is back within 3 dB of the floor at rest;
    -1 if it never gets there before the run ends."""
    db = 10*np.log10(misalignment[N:])
    back = db < floor + 3
    if not back.any():
        return -1
    return int(np.argmax(back))
# === end of the copied block ===


# === copied from bank-skf-fkf.ipynb sections 7 and 7.2 (commit 57bd568), NOT edited ===
def time_to(misalignment, level_db):
    """First step after the change below level_db; -1 if never."""
    below = 10*np.log10(misalignment[N:]) < level_db
    return int(np.argmax(below)) if below.any() else -1


LEVELS = [0.0, -5.0, -10.0, -15.0]


def first_below(misalignment, level_db):
    below = 10*np.log10(misalignment) < level_db
    return int(np.argmax(below)) if below.any() else -1


def measure(rest_curve, change_curve):
    floor, reached = steady_state(rest_curve)
    return dict(floor=floor, reached=reached, settled=is_settled(rest_curve, DRIFT_DB),
                to_17=first_below(rest_curve, TARGET_DB + 3), to_20=first_below(rest_curve, TARGET_DB),
                levels=[time_to(change_curve, level) for level in LEVELS], rec=recovery(change_curve, floor),
                back_20=time_to(change_curve, TARGET_DB), mean_db=10*np.log10(change_curve.mean()),
                curve=change_curve)


def show_table(rows):
    print(f"{'':<34}{'at rest':-^46}  {'after the flip':-^50}")
    print(f"{'filter':<22}{'eps or v':>12}{'floor':>8}{'settled':>9}{'own+3':>7}{'-17 dB':>8}{'-20 dB':>8}"
          + "".join(f"{level:>8.0f}" for level in LEVELS) + f"{'own+3':>9}{'-20 dB':>9}{'mean [dB]':>11}")
    for name, r in rows.items():
        value = "grid" if np.isnan(r["value"]) else f"{r['value']:.4e}"
        print(f"{name:<22}{value:>12}{r['floor']:>8.2f}{'yes' if r['settled'] else 'NO':>9}{r['reached']:>7d}"
              f"{r['to_17']:>8d}{r['to_20']:>8d}" + "".join(f"{s:>8d}" for s in r["levels"])
              + f"{r['rec']:>9d}{r['back_20']:>9d}{r['mean_db']:>11.2f}")
    print("   steps; -1 = never within the run")


EPS_GRID = np.logspace(-8, -3, 11)          # notebook 09


def pick(grid, floors, settled, target):
    values, kept = grid[settled], floors[settled]
    if target < kept.min():
        return values[np.argmin(kept)], "target below every settled floor"
    if target > kept.max():
        return values[np.argmax(kept)], "target above every settled floor"
    order = np.argsort(kept)
    return 10**np.interp(target, kept[order], np.log10(values)[order]), "ok"
# === end of the copied block ===


def sweep_kf(b_eta, X, D):
    # notebook 09's search for the KF, as sweep() of the bank notebook, one grid value at a time
    floors, settled = [], []
    for eps in EPS_GRID:
        curve = KF_batch(X, D, ho, b_eta, np.full(len(X), eps))[0].mean(axis=0)
        floors.append(steady_state(curve)[0])
        settled.append(is_settled(curve, DRIFT_DB))
    return np.array(floors), np.array(settled)


SMOOTH = 101                # centred moving average on both means, in steps


def smoothed_ratio(num, den, cut=None):
    """Moving average of both means, then the ratio. No window crosses step cut (the change)."""
    if cut is not None:
        return np.concatenate([smoothed_ratio(num[:cut], den[:cut]), smoothed_ratio(num[cut:], den[cut:])])
    return uniform_filter1d(num, SMOOTH, mode="nearest")/uniform_filter1d(den, SMOOTH, mode="nearest")


def measure_excess(s_rest, s_change, den_rest, den_change):
    """s_rest (R, N), s_change (R, 2N); den_* = E[(x_t' theta_t)^2], the mean over realisations at each step."""
    num_rest, num_change = (s_rest**2).mean(axis=0), (s_change**2).mean(axis=0)
    rest_curve, change_curve = smoothed_ratio(num_rest, den_rest), smoothed_ratio(num_change, den_change, cut=N)
    tail = slice(3*N//4, N)                            # the window of steady_state
    return dict(floor=10*np.log10(num_rest[tail].mean()/den_rest[tail].mean()),
                to_17=first_below(rest_curve, TARGET_DB + 3), to_20=first_below(rest_curve, TARGET_DB),
                levels=[time_to(change_curve, level) for level in LEVELS + [TARGET_DB]],
                mean_db=10*np.log10(num_change.mean()/den_change.mean()),
                rest_curve=rest_curve, curve=change_curve)


def show_excess(rows):
    print(f"{'':<34}{'at rest':-^37}  {'after the flip, excess error':-^40}")
    print(f"{'filter':<22}{'eps or v':>12}{'mis.':>8}{'excess':>8}{'gap':>7}{'-17 dB':>7}{'-20 dB':>7}"
          + "".join(f"{level:>8.0f}" for level in LEVELS + [TARGET_DB]) + f"{'mean mis.':>11}{'mean exc.':>11}")
    for name, r in rows.items():
        x = r["excess"]
        value = "grid" if np.isnan(r["value"]) else f"{r['value']:.4e}"
        print(f"{name:<22}{value:>12}{r['floor']:>8.2f}{x['floor']:>8.2f}{x['floor'] - r['floor']:>7.2f}"
              f"{x['to_17']:>7d}{x['to_20']:>7d}" + "".join(f"{s:>8d}" for s in x["levels"])
              + f"{r['mean_db']:>11.2f}{x['mean_db']:>11.2f}")
    print("   floors and means in dB; mis. = misalignment, excess = excess error, gap = excess - mis.")
    print("   steps on the excess error (smoothed, good to about 50 steps); -1 = never within the run")


def run_filter(name, value, b_eta, X, D, flip_at=None):
    # misalignment and s_t, (B, n) each
    family, method = name.split(", ")
    if method == "bank":
        out = bank_batch(X, D, ho, EPS_BANK if family == "sKF" else V_BANK, Q_BANK, family, chi_laplacian,
                         log_density_laplacian, b_eta, flip_at=flip_at, excess=True)
        return out["mis"], out["s"]
    values = np.full(len(X), value)
    if family == "sKF":
        return sKF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    if family == "fKF":
        return fKF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    mis, _, s = KF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    return mis, s


# floor, own + 3, recovery and mean over the record printed by bank-skf-fkf.ipynb (commit 57bd568)
BANK_NB = {(-0.9, "sKF, exact"): (-20.01, 4756, 7584, -14.08), (-0.9, "sKF, bank"): (-25.92, 68043, 75315, -13.89),
           (-0.9, "fKF, exact"): (-19.99, 5461, 8188, -13.45), (-0.9, "fKF, bank"): (-15.68, 28234, 29311, -11.92),
           (0.0, "sKF, exact"): (-20.01, 1564, 1221, -17.22), (0.0, "sKF, bank"): (-44.92, 6963, 6433, -22.64),
           (0.0, "fKF, exact"): (-20.00, 1056, 1596, -16.67), (0.0, "fKF, bank"): (-44.58, 12135, 12233, -23.01)}


def scenario_run(ar_a, tuned):
    """The KF tuned to -20 dB, then the five filters at rest and across the change, with s_t."""
    start = time.time()
    (X_r, D_r), (X_c, D_c) = scenario_signals(ar_a)
    b_eta = noise_of(ar_a)[1]
    den_r = (clean_output(X_r)**2).mean(axis=0)
    den_c = (clean_output(X_c, change_at=N)**2).mean(axis=0)
    print(f"mean of (x_t' theta_t)^2 over the change runs: {den_c[M:].mean():.4f}; "
          f"h' R_xx h = {signal_power(ar_a):.4f}")
    floors, settled = sweep_kf(b_eta, X_r, D_r)
    eps_kf, status = pick(EPS_GRID, floors, settled, TARGET_DB)
    print("KF: grid floors " + " ".join(f"{f:.1f}" + ("" if s else "*") for f, s in zip(floors, settled))
          + "   (* not settled)")
    print(f"   at -20 dB: eps = {eps_kf:.4e}, {status}   [{time.time() - start:.0f} s]")
    values = {"sKF, exact": tuned["sKF"], "sKF, bank": np.nan, "fKF, exact": tuned["fKF"], "fKF, bank": np.nan,
              "KF, exact": eps_kf}
    rows = {}
    for name, value in values.items():
        mis_r, s_r = run_filter(name, value, b_eta, X_r, D_r)
        mis_c, s_c = run_filter(name, value, b_eta, X_c, D_c, flip_at=N)
        rows[name] = dict(value=value, rest_curve=mis_r.mean(axis=0), **measure(mis_r.mean(axis=0), mis_c.mean(axis=0)),
                          excess=measure_excess(s_r, s_c, den_r, den_c))
        print(f"{name} done [{time.time() - start:.0f} s]")
    print("\ncontrol: the four filters of the bank notebook against its printed numbers (ours / printed)")
    for name, r in rows.items():
        if (ar_a, name) in BANK_NB:
            floor, steps, rec, mean = BANK_NB[(ar_a, name)]
            print(f"   {name:<11} floor {r['floor']:.2f} / {floor:.2f}, own+3 {r['reached']} / {steps}, "
                  f"recovery {r['rec']} / {rec}, mean {r['mean_db']:.2f} / {mean:.2f}")
    print("\nmisalignment")
    show_table(rows)
    print("\nexcess error")
    show_excess(rows)
    return rows


print("measurement ready")

### 5.1 The sweep

- `sweep`: the sKF and fKF searches of the bank notebook (section 7.2, copied). With `pick` and `sweep_kf` they tune
  the single filters to $-20$ dB under Gaussian noise.
- `filter_run`: one filter in one scenario, rest run and change run, summarised by `summarise` and saved.
  - **mean**: the mean over the realisations at each step, measured as in the earlier notebooks.
  - **median**: the median at each step, of the misalignment and of each realisation's smoothed $s_t^2$, measured by
    the same functions. The 25 % and 75 % curves are kept for the figures.
  - Kept: the curves from 0.25 s before the change to 2.5 s after it, at full resolution, the rest run every 4th step,
    and the banks' mean weights every 8th step. Not the per-realisation arrays.
- `scenario_sweep`: the 18 filters of one scenario. At $\beta^* = 0.2$ it compares the filters run before with their
  printed numbers (the control).
- `show_sweep`: the table of one scenario, from the mean or the median curves.

In [ ]:
Q_SWEEP = [1e-4, 1e-3, 1e-2, 1e-1, 1/3]       # 1e-4 is the value of every earlier bank run
V_GRID = np.logspace(-6, -1, 11)               # fkf.ipynb, leszek-setup.ipynb; the fKF search of bank-skf-fkf.ipynb


# === copied from bank-skf-fkf.ipynb section 7.2 (commit 57bd568), NOT edited ===
def sweep(family, chi_fn, scale, grid, X, D):
    # notebook 09's search, as clipped-matched.ipynb section 5: every grid point at R realisations at once
    g = len(grid)
    Xg, Dg, values = np.repeat(X, g, axis=0), np.repeat(D, g, axis=0), np.tile(grid, len(X))
    mis = (sKF_batch(Xg, Dg, ho, scale, values, chi_fn) if family == "sKF"
           else fKF_batch(Xg, Dg, ho, scale, values, chi_fn))
    curves = mis.reshape(len(X), g, -1).mean(axis=0)
    del mis
    return (np.array([steady_state(c)[0] for c in curves]),
            np.array([is_settled(c, DRIFT_DB) for c in curves]))
# === end of the copied block ===


# The single filters at beta* = 0.2, tuned to -20 dB in the earlier notebooks and given here, not re-tuned: sKF and fKF
# from bank-skf-fkf.ipynb, the KF from bank-kf-excess-error.ipynb (printed there to five digits).
TUNED_HEAVY = {-0.9: {"sKF": 1.3493e-6, "fKF": 2.985e-4, "KF": 8.8577e-7},
               0.0: {"sKF": 1.319e-6, "fKF": 3.017e-4, "KF": 1.6553e-6}}
# floor, own+3, recovery and mean over the record printed by bank-kf-excess-error.ipynb (commit f2ded06)
PREVIOUS = dict(BANK_NB)
PREVIOUS.update({(-0.9, "KF, exact"): (-20.01, 1254, 1972, -15.63), (0.0, "KF, exact"): (-20.02, 603, 791, -17.65)})

CACHE = Path("results/kf-bank-q-sweep")        # one file per run, written as it finishes; delete it to rerun all
WINDOW = slice(N - 2000, N + 20000)            # the change run kept at full resolution: -0.25 s to +2.5 s
THIN = 4                                       # the rest run kept every THIN steps


def cached(key, compute):
    path = CACHE/f"{key}.pkl"
    if path.exists():
        with open(path, "rb") as f:
            return pickle.load(f)
    result = compute()
    CACHE.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)
    return result


def tuned_values(ar_a, beta, b_eta, X, D):
    """The single filters' -20 dB values: given at beta* = 0.2, from notebook 09's search at any other beta*
    (the searches of the bank notebook: settled points only, interpolated)."""
    if beta == BETA:
        return dict(TUNED_HEAVY[ar_a])

    def search():
        found = {}
        for family, grid in (("sKF", EPS_GRID), ("fKF", V_GRID), ("KF", EPS_GRID)):
            floors, settled = (sweep_kf(b_eta, X, D) if family == "KF"
                               else sweep(family, chi_laplacian, b_eta, grid, X, D))
            if settled.any():
                value, status = pick(grid, floors, settled, TARGET_DB)
            else:                                      # pick needs one settled point; say so instead of failing
                value, status = pick(grid, floors, np.ones(len(grid), dtype=bool), TARGET_DB)
                status = "NO grid point settled; picked among all of them, " + status
            found[family] = dict(floors=floors, settled=settled, value=value, status=status)
        return found
    found = cached(f"search_ar{ar_a:g}_beta{beta:g}", search)
    for family, r in found.items():
        print(f"{family}: grid floors " + " ".join(f"{f:.1f}" + ("" if s else "*") for f, s in
                                                     zip(r["floors"], r["settled"])) + "   (* not settled)")
        print(f"   at -20 dB: {r['value']:.4e}, {r['status']}")
    return {family: r["value"] for family, r in found.items()}


def run_one(name, value, q, b_eta, X, D, flip_at=None):
    """Misalignment and s_t (B, n) of one filter, and its weights averaged over the rows (K, n) if it is a bank.
    run_filter of bank-kf-excess-error.ipynb, with the banks' q as a parameter and the bank of KFs added."""
    family, method = name.split(", ")
    if method == "bank":
        out = (kf_bank_batch(X, D, ho, EPS_BANK, q, chi_laplacian, log_density_laplacian, b_eta, flip_at=flip_at,
                             excess=True) if family == "KF" else
               bank_batch(X, D, ho, EPS_BANK if family == "sKF" else V_BANK, q, family, chi_laplacian,
                          log_density_laplacian, b_eta, flip_at=flip_at, excess=True))
        return out["mis"], out["s"], out["mu"]
    values = np.full(len(X), value)
    if family == "sKF":
        mis, s = sKF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    elif family == "fKF":
        mis, s = fKF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    else:
        mis, _, s = KF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    return mis, s, None


def smooth_split(a, cut=None):
    """The 101-step centred moving average of smoothed_ratio, along the last axis; no window crosses step cut."""
    if cut is not None:
        return np.concatenate([smooth_split(a[..., :cut]), smooth_split(a[..., cut:])], axis=-1)
    return uniform_filter1d(a, SMOOTH, axis=-1, mode="nearest")


def measure_excess_median(num_rest, num_change, den_rest, den_change):
    """measure_excess for the median: num_* are the medians over the realisations of each realisation's s_t^2,
    already smoothed (the median of 20 raw squares is noisy and reads low); den_* the means of (x_t' theta_t)^2."""
    rest_curve, change_curve = num_rest/smooth_split(den_rest), num_change/smooth_split(den_change, cut=N)
    tail = slice(3*N//4, N)                            # the window of steady_state
    return dict(floor=10*np.log10(num_rest[tail].mean()/den_rest[tail].mean()),
                to_17=first_below(rest_curve, TARGET_DB + 3), to_20=first_below(rest_curve, TARGET_DB),
                levels=[time_to(change_curve, level) for level in LEVELS + [TARGET_DB]],
                mean_db=10*np.log10(num_change.mean()/den_change.mean()),
                rest_curve=rest_curve, curve=change_curve)


def summarise(mis_r, s_r, mis_c, s_c, mu_c, den_r, den_c):
    """Numbers and curves of one filter in one scenario, from its rest run (R, N) and its change run (R, 2N).
    mean: the mean over the realisations at each step, measured as in the earlier notebooks (measure,
    measure_excess). median: the median over the realisations at each step, of the misalignment and of each
    realisation's s_t^2 smoothed over 101 steps, measured by the same functions. The 25 % and 75 % curves are kept
    for the bands of the figures."""
    pct = (25, 50, 75)
    mis_q_r, mis_q_c = np.percentile(mis_r, pct, axis=0), np.percentile(mis_c, pct, axis=0)
    sm_r, sm_c = smooth_split(s_r**2), smooth_split(s_c**2, cut=N)
    exc_q_r, exc_q_c = np.percentile(sm_r, pct, axis=0), np.percentile(sm_c, pct, axis=0)
    mean_r, mean_c = mis_r.mean(axis=0), mis_c.mean(axis=0)
    out = {"mean": measure(mean_r, mean_c), "median": measure(mis_q_r[1], mis_q_c[1]),
           "mean_x": measure_excess(s_r, s_c, den_r, den_c),
           "median_x": measure_excess_median(exc_q_r[1], exc_q_c[1], den_r, den_c)}
    den_sm_r, den_sm_c = smooth_split(den_r), smooth_split(den_c, cut=N)
    f32 = lambda a: np.asarray(a, dtype=np.float32)
    out["curves"] = {
        "mis_mean": f32(mean_c[WINDOW]), "mis_q": f32(mis_q_c[:, WINDOW]),
        "exc_mean": f32(out["mean_x"]["curve"][WINDOW]), "exc_q": f32((exc_q_c/den_sm_c)[:, WINDOW]),
        "rest_mis_mean": f32(mean_r[::THIN]), "rest_mis_q": f32(mis_q_r[:, ::THIN]),
        "rest_exc_mean": f32(out["mean_x"]["rest_curve"][::THIN]), "rest_exc_q": f32((exc_q_r/den_sm_r)[:, ::THIN]),
        "mu": None if mu_c is None else f32(mu_c[:, ::8])}
    for key in ("mean", "median", "mean_x", "median_x"):         # the full-length curves are not kept
        out[key].pop("curve", None)
        out[key].pop("rest_curve", None)
    return out


def filter_run(name, value, q, ar_a, beta, sig, b_eta, den):
    """One filter in one scenario, rest and change runs, summarised; read from the cache when it has been run."""
    tag = "grid" if np.isnan(value) else f"{value:.6e}"
    key = f"{name.replace(', ', '-')}_{tag}_q{'-' if q is None else f'{q:g}'}_ar{ar_a:g}_beta{beta:g}"

    def compute():
        start = time.time()
        (X_r, D_r), (X_c, D_c) = sig
        mis_r, s_r, _ = run_one(name, value, q, b_eta, X_r, D_r)
        mis_c, s_c, mu_c = run_one(name, value, q, b_eta, X_c, D_c, flip_at=N)
        out = summarise(mis_r, s_r, mis_c, s_c, mu_c, *den)
        out.update(name=name, value=value, q=q, ar_a=ar_a, beta=beta, seconds=time.time() - start)
        return out
    return cached(key, compute)


def show_sweep(rows, stat):
    """The numbers of one scenario, from the mean curves (stat = "mean") or the median curves (stat = "median")."""
    print(f"{'':<22}{' misalignment, ' + stat + ' ':-^72}  {' excess error, ' + stat + ' ':-^38}")
    print(f"{'filter':<12}{'q':>10}{'floor':>8}{'sett.':>6}{'own+3':>7}{'-17':>7}{'0':>7}{'-10':>7}{'-15':>7}"
          f"{'rec.':>7}{'-20':>7}{'mean':>8}  {'floor':>7}{'-10':>7}{'-15':>7}{'-20':>7}{'mean':>8}")
    for (name, q), r in rows.items():
        m, x = r[stat], r[stat + "_x"]
        print(f"{name:<12}{'' if q is None else f'{q:.3g}':>10}{m['floor']:>8.2f}{'yes' if m['settled'] else 'NO':>6}"
              f"{m['reached']:>7d}{m['to_17']:>7d}{m['levels'][0]:>7d}{m['levels'][2]:>7d}{m['levels'][3]:>7d}"
              f"{m['rec']:>7d}{m['back_20']:>7d}{m['mean_db']:>8.2f}  {x['floor']:>7.2f}{x['levels'][2]:>7d}"
              f"{x['levels'][3]:>7d}{x['levels'][4]:>7d}{x['mean_db']:>8.2f}")
    print("   floors and means in dB, everything else in steps; own+3 (to the own floor + 3 dB) and -17 from zero at "
          "rest;\n   0, -10, -15, rec. (back to the own floor + 3 dB) and -20 dB after the flip; -1 = never within "
          "the run")


def scenario_sweep(ar_a, beta):
    """The three single filters at -20 dB and the three banks at each q of Q_SWEEP, in one scenario."""
    start = time.time()
    sig = scenario_signals(ar_a, beta)
    (X_r, D_r), (X_c, D_c) = sig
    b_eta = noise_of(ar_a, beta)[1]
    den = ((clean_output(X_r)**2).mean(axis=0), (clean_output(X_c, change_at=N)**2).mean(axis=0))
    print(f"b_eta = E|eta| = {b_eta:.4e}; mean of (x_t' theta_t)^2 over the change runs {den[1][M:].mean():.4f}, "
          f"h' R_xx h = {signal_power(ar_a):.4f}")
    tuned = tuned_values(ar_a, beta, b_eta, X_r, D_r)
    rows = {}
    for family in ("sKF", "fKF", "KF"):
        rows[(f"{family}, exact", None)] = filter_run(f"{family}, exact", tuned[family], None, ar_a, beta, sig, b_eta,
                                                      den)
    for q in Q_SWEEP:
        for family in ("sKF", "fKF", "KF"):
            rows[(f"{family}, bank", q)] = filter_run(f"{family}, bank", np.nan, q, ar_a, beta, sig, b_eta, den)
        print(f"q = {q:.3g} done [{time.time() - start:.0f} s]", flush=True)
    if beta == BETA:
        print("\ncontrol: the filters run before, against their printed numbers (ours / printed)")
        for (name, q), r in rows.items():
            if (ar_a, name) in PREVIOUS and q in (None, Q_BANK):
                floor, steps, rec, mean = PREVIOUS[(ar_a, name)]
                m = r["mean"]
                print(f"   {name:<11} floor {m['floor']:.2f} / {floor:.2f}, own+3 {m['reached']} / {steps}, "
                      f"recovery {m['rec']} / {rec}, mean {m['mean_db']:.2f} / {mean:.2f}")
    print()
    show_sweep(rows, "mean")
    print()
    show_sweep(rows, "median")
    print(f"[{time.time() - start:.0f} s; the 18 runs took {sum(r['seconds'] for r in rows.values())/60:.0f} min when they were run (a run read from results/ keeps its own time)]")
    return rows


results = {}
print("sweep ready")

## 6. AR($-0.9$), $\beta^* = 0.2$

The scenario of the earlier notebooks. Control: the exact sKF and fKF and the two banks at $q = 10^{-4}$ must give the numbers printed by `bank-kf-excess-error.ipynb`; the KF too, up to its $\varepsilon$ being given here to five digits.

In [ ]:
results[(-0.9, BETA)] = scenario_sweep(-0.9, BETA)

## 7. White input, $\beta^* = 0.2$

The same control.

In [ ]:
results[(0.0, BETA)] = scenario_sweep(0.0, BETA)

## 8. AR($-0.9$), Gaussian noise ($\beta^* = 2$)

The single filters tuned again to $-20$ dB (the search is printed first; it is saved like the runs).

In [ ]:
results[(-0.9, BETA_GAUSS)] = scenario_sweep(-0.9, BETA_GAUSS)

## 9. White input, Gaussian noise ($\beta^* = 2$)

As above.

In [ ]:
results[(0.0, BETA_GAUSS)] = scenario_sweep(0.0, BETA_GAUSS)

## 10. Figures

**Figure 1 (request 3).** The six filters at $q = 10^{-4}$, $\beta^* = 0.2$: mean against median with its 25-75 % band,
misalignment and excess error, across the change. Colour = family (sKF orange, fKF aqua, KF blue); solid = bank,
dashed = single filter at $-20$ dB. Grey: $-20$ dB; black: the change.

**Figures 2-5 (requests 2 and 4).** Each bank at the five values of $q$ (light blue $10^{-4}$ to dark blue $1/3$),
median and 25-75 % band, with its single filter (grey, dashed): at $-20$ dB, except with AR($-0.9$) input and
Gaussian noise, where no settled grid point reaches it (section 8). Misalignment at $\beta^* = 0.2$ and 2, then the
excess error at both.

**Figures 6-7.** From the median curves, against $q$: the floor at rest, the steps to $-15$ dB after the change, and
the mean over the record (the draft's metric). Misalignment, then the excess error. Dashed: the single filter.
Every plotted number is also in the tables of sections 6-9.

In [ ]:
# Colours checked with the palette validator of the dataviz skill (white surface): the three families take
# categorical slots 1-3 in that order (all pairs pass; the KF keeps the blue of the earlier notebooks), and q takes a
# five-step ordinal blue ramp, light for small q to dark for large q (passes). The aqua of the fKF is below 3:1
# contrast on white, so every plotted number is also in the tables of sections 6-9.
FAMILY_COLOUR = {"KF": "#2a78d6", "sKF": "#eb6834", "fKF": "#1baf7a"}
Q_COLOUR = dict(zip(Q_SWEEP, ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#0d366b"]))
SINGLE_GREY, TARGET_GREY, GRID_GREY, INK = "#52514e", "#a3a29b", "#e1e0d9", "#0b0b0b"
SCENARIOS = [(-0.9, BETA), (0.0, BETA), (-0.9, BETA_GAUSS), (0.0, BETA_GAUSS)]
T_MAX = {-0.9: 2.5, 0.0: 0.6}                  # seconds after the change shown, as in the earlier notebooks


def scenario_label(ar_a, beta):
    return f"{'AR(-0.9)' if ar_a else 'white'}, beta* = {beta:g}"


def window_time(curve):
    # time from the change [s] of the kept window of the change run
    return (np.arange(curve.shape[-1]) + WINDOW.start - N)/FS


def style_axes(ax, t_max, ylim):
    ax.axvline(0, color=INK, lw=0.8)                          # the change
    ax.axhline(TARGET_DB, color=TARGET_GREY, lw=0.8)          # -20 dB
    ax.set(xlim=(-0.25, t_max), ylim=ylim)
    ax.grid(color=GRID_GREY, lw=0.6)
    ax.set_axisbelow(True)


def db_range(curves, top=8.0):
    # y range of a row: from below the lowest median (5 dB steps) to top
    low = min(float(np.nanmin(10*np.log10(np.maximum(c, 1e-30)))) for c in curves)
    return (5*np.floor(low/5) - 2, top)


def figure_mean_median(results):
    """Figure 1: the six filters at q = 1e-4, beta* = 0.2, mean against median."""
    fig, axes = plt.subplots(2, 4, figsize=(20, 8.6), constrained_layout=True, sharey="row")
    panels = (("mis", "mean"), ("mis", "median"), ("exc", "mean"), ("exc", "median"))
    for row, ar_a in enumerate((-0.9, 0.0)):
        rows = {key: r for key, r in results[(ar_a, BETA)].items() if key[1] in (None, Q_BANK)}
        some = next(iter(rows.values()))["curves"]["mis_mean"]
        t = window_time(some)
        keep = t <= T_MAX[ar_a]
        ylim = db_range([r["curves"][key][keep] for r in rows.values() for key in ("mis_mean", "exc_mean")]
                        + [r["curves"][key][1][keep] for r in rows.values() for key in ("mis_q", "exc_q")])
        for col, (metric, stat) in enumerate(panels):
            ax = axes[row, col]
            for (name, q), r in rows.items():
                family, method = name.split(", ")
                c, colour = r["curves"], FAMILY_COLOUR[family]
                label = f"{family} bank, q = {q:g}" if method == "bank" else f"{family}, exact, -20 dB"
                if stat == "median":
                    lo, mid, hi = c[f"{metric}_q"]
                    ax.fill_between(t[keep], 10*np.log10(lo[keep]), 10*np.log10(hi[keep]), color=colour, alpha=0.14,
                                    lw=0)
                else:
                    mid = c[f"{metric}_mean"]
                ax.plot(t[keep], 10*np.log10(mid[keep]), color=colour, ls="-" if method == "bank" else "--", lw=1.5,
                        label=label)
            style_axes(ax, T_MAX[ar_a], ylim)
            what = "misalignment" if metric == "mis" else "excess error"
            ax.set_title(f"{'AR(-0.9)' if ar_a else 'white input'}, {what}, "
                         + ("mean" if stat == "mean" else "median, 25-75 % band"), fontsize=10)
            ax.set_xlabel("time from the change [s]")
        axes[row, 0].set_ylabel("[dB]")
    axes[0, 0].legend(fontsize=8, loc="upper right")
    fig.suptitle(r"$q = 10^{-4}$, $\beta^* = 0.2$, 5 dB: the mean over the 20 realisations against their median")
    plt.show()


def figure_q(results, beta, metric):
    """Figures 2-5: each bank at the five q, median and 25-75 % band, with its single filter at -20 dB (median)."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 9), constrained_layout=True, sharey="row")
    for row, ar_a in enumerate((-0.9, 0.0)):
        rows = results[(ar_a, beta)]
        t = window_time(rows[("sKF, exact", None)]["curves"]["mis_mean"])
        keep = t <= T_MAX[ar_a]
        ylim = db_range([r["curves"][f"{metric}_q"][0][keep] for r in rows.values()])
        for col, family in enumerate(("sKF", "fKF", "KF")):
            ax = axes[row, col]
            single = rows[(f"{family}, exact", None)]["curves"][f"{metric}_q"][1]
            ax.plot(t[keep], 10*np.log10(single[keep]), color=SINGLE_GREY, ls="--", lw=1.5,
                    label=f"{family}, exact")
            for q in Q_SWEEP:
                lo, mid, hi = rows[(f"{family}, bank", q)]["curves"][f"{metric}_q"]
                ax.fill_between(t[keep], 10*np.log10(lo[keep]), 10*np.log10(hi[keep]), color=Q_COLOUR[q], alpha=0.14,
                                lw=0)
                ax.plot(t[keep], 10*np.log10(mid[keep]), color=Q_COLOUR[q], lw=1.5, label=f"bank, q = {q:.3g}")
            style_axes(ax, T_MAX[ar_a], ylim)
            ax.set_title(f"{family} bank, {'AR(-0.9)' if ar_a else 'white input'}", fontsize=10)
            ax.set_xlabel("time from the change [s]")
            if row == 0:
                ax.legend(fontsize=8, loc="upper right")
        axes[row, 0].set_ylabel(("misalignment" if metric == "mis" else "excess error") + ", median [dB]")
    fig.suptitle(f"The banks at five q, beta* = {beta:g}, 5 dB: "
                 + ("misalignment" if metric == "mis" else "excess error") + ", median and 25-75 % band")
    plt.show()


def summary_value(r, metric, key):
    m = r["median"] if metric == "mis" else r["median_x"]
    if key == "to15":
        steps = m["levels"][3]                  # -15 dB after the flip (LEVELS, and LEVELS + [-20] for the excess)
        return np.nan if steps < 0 else steps
    return m[key]


def figure_summary(results, metric):
    """Figures 6-7: floor at rest, steps to -15 dB after the flip, mean over the record, against q (medians)."""
    fig, axes = plt.subplots(3, 4, figsize=(19, 11), constrained_layout=True, sharex=True)
    measures = (("floor", "floor at rest [dB]"), ("to15", "steps to -15 dB after the flip"),
                ("mean_db", "mean over the record [dB]"))
    for col, (ar_a, beta) in enumerate(SCENARIOS):
        rows = results[(ar_a, beta)]
        for row, (key, label) in enumerate(measures):
            ax = axes[row, col]
            for family in ("sKF", "fKF", "KF"):
                colour = FAMILY_COLOUR[family]
                ax.axhline(summary_value(rows[(f"{family}, exact", None)], metric, key), color=colour, ls="--", lw=1.2,
                           label=f"{family}, exact")
                ax.plot(Q_SWEEP, [summary_value(rows[(f"{family}, bank", q)], metric, key) for q in Q_SWEEP],
                        color=colour, lw=1.6, marker="o", ms=6, mec="white", mew=1.5, label=f"{family} bank")
            ax.set_xscale("log")
            if key == "to15":
                reached = [y for line in ax.get_lines() for y in np.atleast_1d(line.get_ydata())
                           if np.isfinite(y) and y > 0]
                if reached:
                    ax.set_yscale("log")
                else:                                  # a log axis needs one value
                    ax.text(0.5, 0.5, "never reached", transform=ax.transAxes, ha="center", color=SINGLE_GREY)
            ax.grid(color=GRID_GREY, lw=0.6)
            ax.set_axisbelow(True)
            if row == 0:
                ax.set_title(scenario_label(ar_a, beta), fontsize=10)
            if row == 2:
                ax.set_xlabel("q")
            if col == 0:
                ax.set_ylabel(label)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside lower center", ncol=6, fontsize=9)
    fig.suptitle(("Misalignment" if metric == "mis" else "Excess error")
                 + " against q, from the median curves; dashed: the single filter; a gap = never reached")
    plt.show()


figure_mean_median(results)

In [ ]:
figure_q(results, BETA, "mis")

In [ ]:
figure_q(results, BETA_GAUSS, "mis")

In [ ]:
figure_q(results, BETA, "exc")

In [ ]:
figure_q(results, BETA_GAUSS, "exc")

In [ ]:
figure_summary(results, "mis")

In [ ]:
figure_summary(results, "exc")

## 11. Findings

**Checks: all pass** (section 4).
1. The numba code against Algorithm 3 in plain numpy. Gaussian $\chi$: outputs to $10^{-13}$, $\log\mu$ to $5 \times 10^{-12}$.
   Laplacian $\chi$: $6.5 \times 10^{-7}$ and $4.7 \times 10^{-5}$, less than the reference moves when its own input changes by
   one unit in the last place ($1.2 \times 10^{-6}$ and $8.8 \times 10^{-5}$). With the Laplacian likelihood these filters
   amplify rounding: at the first updates $k_t \approx 125$, where `log_mills` subtracts two terms of about 7800. So bit for
   bit is not a meaningful test for this bank; agreement within the one-ulp yardstick is.
2. One filter ($K = 1$) is `KF_batch`, within its yardstick ($2.3 \times 10^{-6}$ against $2.4 \times 10^{-6}$).
3. $q = 0$: each filter is `KF_batch` at its own $\varepsilon$ (to $9.0 \times 10^{-5}$, the yardstick being $9.0 \times 10^{-5}$),
   and the weights are eq. (58) to $3 \times 10^{-11}$.
4. From a scalar prior the first update is the sKF bank's (locations to $4 \times 10^{-9}$, weights to $10^{-12}$,
   $\mathrm{tr}(\boldsymbol{\Sigma}_{1,j})/M = v_{1,j}$ to $10^{-10}$), and the second mixing gives $\bar{v}_{2,j}$ of eq. (54) to
   $8 \times 10^{-11}$: the spread term of the mixing is right.
5. With the Gaussian $\chi$ the bank is the textbook IMM, written without reference to Algorithm 3, to $2 \times 10^{-15}$.
6. Brute force over the $\varepsilon$ sequences: exact at the first update and at $q = 0$; the merge costs $1.1 \times 10^{-4}$
   in the weights at $q = 10^{-4}$ and $1.4 \times 10^{-2}$ at $q = 0.2$, the size it had for the sKF bank.
7. **Anchor: Leszek's bank of full KFs is reproduced.** Best fixed $\varepsilon$ $-8.17$ dB, bank $-10.69$ dB (with and
   without his clamp), against $-8.2$ and $-10.7$ in the draft's preliminary-results remark.
8. Control: with $\beta^* = 0.2$ the five filters run before give their printed floor, steps, recovery and mean
   exactly, in both scenarios.

**1. The bank of KFs: with coloured input it is the best filter by far.**

| AR($-0.9$), $\beta^* = 0.2$, $q = 10^{-4}$ | floor at rest [dB] | $-15$ dB after the flip [steps] | mean over the record [dB] | excess error, mean over the record [dB] |
|---|---|---|---|---|
| **KF bank** | $\mathbf{-41.87}$ | **729** | $\mathbf{-20.55}$ | $\mathbf{-25.99}$ |
| KF at $-20$ dB | $-20.01$ | 1 738 | $-15.63$ | $-20.79$ |
| exact sKF at $-20$ dB | $-20.01$ | 6 497 | $-14.08$ | $-17.01$ |
| sKF bank | $-25.92$ | 31 119 | $-13.89$ | $-20.72$ |
| fKF bank | $-15.68$ | 72 194 | $-11.92$ | $-17.63$ |

- It recovers from the change 2.4 times faster than the KF and settles 22 dB lower, so its mean over the record is
  4.9 dB better than the KF's and 6.7 dB better than the sKF bank's. With coloured input the full covariance gives the
  bank what the sKF and fKF banks lacked, as the draft's "decorrelated input" paragraph says.
- With white input it is about 1 dB better than the other two banks ($-23.92$ dB against $-22.64$ and $-23.01$), at
  the same floor ($-45$ dB).
- Cost: 10 min for a rest and change run, against 3.6 min for one KF (`KF_batch`) and 1.5 / 1.7 min for the sKF / fKF
  banks.

**2. Larger $q$: it helps only with AR input, and clearly only for the sKF bank with heavy-tailed noise.** Mean over the record in dB, from the mean curves (the
median curves give the same best $q$ in 11 rows of 12 and a tie within 0.01 dB in the 12th); bold: the best $q$;
single: the single filter of the same family (* not at $-20$ dB, see 4).

| scenario | bank | $10^{-4}$ | $10^{-3}$ | $10^{-2}$ | $10^{-1}$ | $1/3$ | single |
|---|---|---|---|---|---|---|---|
| AR, $\beta^* = 0.2$ | sKF | $-13.89$ | $-13.72$ | $-15.28$ | $\mathbf{-16.52}$ | $-12.31$ | $-14.08$ |
| | fKF | $\mathbf{-11.92}$ | $-10.23$ | $-10.64$ | $-7.40$ | $-5.97$ | $-13.45$ |
| | KF | $-20.55$ | $\mathbf{-20.57}$ | $-20.49$ | $-19.05$ | $-12.26$ | $-15.63$ |
| white, $\beta^* = 0.2$ | sKF | $\mathbf{-22.64}$ | $-22.64$ | $-22.57$ | $-20.49$ | $-11.67$ | $-17.22$ |
| | fKF | $\mathbf{-23.01}$ | $-22.51$ | $-19.84$ | $-6.91$ | $-6.61$ | $-16.67$ |
| | KF | $-23.92$ | $\mathbf{-23.94}$ | $-23.87$ | $-21.94$ | $-13.20$ | $-17.65$ |
| AR, $\beta^* = 2$ | sKF | $-8.68$ | $-8.78$ | $\mathbf{-10.53}$ | $-10.36$ | $-8.50$ | $-6.77$* |
| | fKF | $-8.00$ | $-7.99$ | $-8.32$ | $-8.53$ | $\mathbf{-9.46}$ | $-8.00$* |
| | KF | $\mathbf{-14.09}$ | $-13.03$ | $-10.83$ | $-6.15$ | $-4.25$ | $-6.79$* |
| white, $\beta^* = 2$ | sKF | $\mathbf{-18.55}$ | $-16.97$ | $-14.68$ | $-10.69$ | $-8.75$ | $-11.56$ |
| | fKF | $\mathbf{-17.09}$ | $-15.52$ | $-14.51$ | $-15.24$ | $-14.48$ | $-10.27$ |
| | KF | $\mathbf{-18.89}$ | $-17.33$ | $-14.98$ | $-10.65$ | $-8.56$ | $-12.06$ |

- **The sKF bank with AR input and heavy-tailed noise improves.** At $q = 0.1$ it no longer crawls after the change:
  $-15$ dB in 5 300 steps against 31 119 at $q = 10^{-4}$ (exact sKF: 6 497), floor $-27.5$ dB, mean $-16.52$ dB, 2.4 dB
  better than the exact sKF. $q = 10^{-2}$ lies between (15 931 steps, $-15.28$ dB).
- **With AR input and Gaussian noise** the sKF and fKF banks also gain from larger $q$, 1.9 and 1.5 dB on the mean
  ($q = 10^{-2}$ and $1/3$), but both stay well behind the KF bank at $q = 10^{-4}$ ($-14.09$ dB), and the fKF bank settles
  at no $q$.
- **Everywhere else the floor pays.** Larger $q$ mixes every filter with its neighbours more often, and the floor rises
  much faster than the recovery speeds up. KF bank, white input, $\beta^* = 0.2$: floors $-45.1$, $-44.7$, $-39.2$, $-26.0$,
  $-13.5$ dB at the five $q$, while $-15$ dB after the flip comes at 477, 472, 467, 493 and 43 737 steps. The fKF bank
  degrades fastest (white input: floor $-44.6$ dB at $q = 10^{-4}$, $-7.0$ at $0.1$).
- **Best $q$:** $10^{-4}$ or $10^{-3}$ in 9 rows of 12. The three exceptions all have AR input: the sKF bank ($0.1$ with
  $\beta^* = 0.2$, $10^{-2}$ with Gaussian noise) and the fKF bank with Gaussian noise ($1/3$). The KF bank never gains:
  its best $q$ beats $10^{-4}$ by 0.02 dB at most. $q = 1/3$, which leaves no persistence, is the worst value in the 11
  other rows.

**3. The median: not too noisy, and it changes no conclusion.**
- From one step to the next the median misalignment moves by about 0.05 dB, against 0.025 dB for the mean (measured
  before this notebook on the four filters of the last one), and the 25-75 % band is narrow: the 20 realisations agree
  closely (Figure 1).
- At $q = 10^{-4}$, $\beta^* = 0.2$ the median and mean floors agree within 0.25 dB, except the fKF bank's, whose short
  spikes the median drops: with white input its floor is $-50.05$ dB from the median against $-44.58$ from the mean, and
  with AR input its excess error $-26.78$ against $-23.99$ dB.
- The record means from the median are up to 1.3 dB lower for the KF and the KF bank with AR input ($-21.82$ against
  $-20.55$ dB for the bank): a few realisations recover more slowly and pull the mean up.
- The excess error needs each realisation's $s_t^2$ smoothed (101 steps) before the median: the median of 20 raw squares
  jitters by 3 dB and reads 3 to 8 dB low.

**4. Gaussian noise ($\beta^* = 2$).**
- **The same SNR is much harder for the Laplacian filters.** The scale they use, $b_\eta = \mathrm{E}\lvert\eta_t\rvert$, is 0.41
  against 0.13 with $\beta^* = 0.2$, at the same variance: the heavy-tailed noise puts most of its power in a few large
  samples, which the bounded score caps, and the Gaussian noise has none. To reach $-20$ dB with white input the single
  filters need a 30 times smaller $\varepsilon$ ($4.5 \times 10^{-8}$ against $1.3 \times 10^{-6}$ for the sKF) and recover from
  the change about 9 times more slowly (sKF 10 601 steps against 1 221; KF 8 652 against 791).
- **With AR($-0.9$) input no single filter reaches $-20$ dB within the run.** The deepest settled floors are $-18.7$ (sKF),
  $-16.4$ (fKF) and $-19.0$ dB (KF, at the lowest $\varepsilon$ of the grid), and these filters recover in 34 000 to 66 000
  steps. In that scenario the banks are compared with slower filters at higher floors, not at an equal floor.
- **The banks gain more over the single filters than with heavy-tailed noise,** because the single filters are slow.
  White input, $q = 10^{-4}$: $-18.55$ / $-17.09$ / $-18.89$ dB for the sKF / fKF / KF bank against $-11.56$ / $-10.27$ /
  $-12.06$ for the single filters at $-20$ dB, 7 dB in each family (5.4 / 6.3 / 6.3 dB with $\beta^* = 0.2$). With AR input
  the KF bank is 7.3 dB better than the KF ($-14.09$ against $-6.79$).
- **But their floors no longer fall far below $-20$ dB:** white input, $q = 10^{-4}$: $-22.1$ / $-19.3$ / $-22.0$ dB, against
  $-44.9$ / $-44.6$ / $-45.1$ with $\beta^* = 0.2$.
- **Larger $q$ helps less still:** with white input every bank is best at $q = 10^{-4}$; with AR input only the sKF bank
  gains, at $q = 10^{-2}$ ($-10.53$ against $-8.68$ dB).

**Speed: why not dask (Augusto's PR #16).** Measured with scratch scripts on the M1 this notebook runs on, not in
this notebook:
- The PR splits the realisations into chunks, one per worker. The filters here already run the 20 realisations side by
  side, so a chunk saves little: the sKF bank takes 0.30 ms a step for 20 realisations and 0.18 ms for 5, so four
  chunks of five on four workers gain 1.7x at most.
- Whole runs at the same time, each in its own process, do help the numpy banks: 3.6x with 4 at once, 4.5x with 8.
- The KF banks, two thirds of the run time, are already parallel inside each run: splitting the cores between runs gives
  1.00x (2 runs x 4 threads), 1.11x (4 x 2) and 1.19x (8 x 1).
- So this notebook would take about 3.9 h instead of 5.2 h, for a new dependency (dask is not installed; the PR's
  requirements.txt does not install on macOS), notebook functions shipped to worker processes, numba threads to pin in
  every worker, and errors that are harder to read. Not worth it for a run that goes overnight. If sweeps like this
  become routine, the levers are a process pool over whole runs of the numpy filters (joblib is lighter than dask for
  that) and a faster kernel for the bank of KFs.

**Not done.**
- The Gaussian likelihood under Gaussian noise, the matched filter there: left out to change one thing at a time.
- Single filters at $-20$ dB with AR input and Gaussian noise: they need a longer run, and the KF a grid below $10^{-8}$.
- The banks' weights against time at larger $q$ (Figure 2 of the bank notebook), which would show why $q = 0.1$ ends the
  sKF bank's crawl with AR input; they are saved with every run (`mu`).
- Which metric to report for the banks with coloured input stays open from the last notebook; both are given here.

In [ ]:
total_runs = sum(r["seconds"] for rows in results.values() for r in rows.values())
print(f"whole notebook: {(time.time() - NOTEBOOK_START)/60:.1f} min; the 72 runs took {total_runs/60:.0f} min "
      f"when they were run")